# Sionna 0.19 — Differentiable Ray Tracing Calibration
## Nottingham Urban Area · Ofcom 2018 · 915.95 MHz

**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15
**Scene:** `scene_with_full_019.xml` — buildings, roads, water, vegetation, trees, railways, barriers
**Dataset:** Ofcom 2018 drive-test — 1 200 receivers, single TX
**Reference:** Hoydis et al. 2023 — *Sionna RT: Differentiable Ray Tracing* (arXiv:2311.18558)

---

## Run Order

Run cells **top to bottom** in this exact sequence every session.

| Step | Cell | Purpose | Time | Output |
|------|------|---------|------|--------|
| 1 | **CELL 0** | Environment setup + imports | <1 min | — |
| 2 | **CELL 1** | Auto-detect scene XML | <1 min | — |
| 3 | **CELL 2** | Global config — frequency, TX power, ray params | <1 min | — |
| 4 | **CELL 3** | Coordinate utilities + DEM elevation | <1 min | — |
| 5 | **CELL 4** | Load scene + configure antennas | 1–2 min | scene object |
| 6 | **CELL 4A** | Assign ITU-R materials — auto-loads calibrated JSON if present | <1 min | materials set |
| 7 | **CELL 6** | Load TX from `transmitter_positions.csv` | <1 min | — |
| 8 | **CELL 7** | Load 1 200 RX from `receiver_locations.csv` | <1 min | receivers list |
| 9 | **CELL 8** | Pre-calibration coverage map (optional visual check) | 5–10 min | map plot |
| 10 | **CELL 8b** | Build calibration targets from Ofcom CSV | <1 min | `calib_rssi_meas` |
| 11 | **CELL 10** | Create trainable material tf.Variables | <1 min | — |
| 12 | **CELL 10b** ★ | **Scalar offset calibration** — global dB shift | ~5 min | `scalar_offset_915mhz.json` |
| 13 | **CELL 11b** ★ | **Material calibration** — ε_r, σ, S per material | ~2–3 h | `calibrated_materials_915mhz.json` |
| 14 | **CELL 15** | Physics-Informed Residual MLP (50 features) | ~1 h | RMSE result |
| 15 | **CELL 16** | MaterialMLP — scene-conditioned end-to-end | ~1 h | MLP weights |

★ = primary calibration cells. Cell 11b requires Cell 10b to have run first.

---

### Quick Test (no 3-hour wait)

Verify scene loads and scalar offset works — skip Cell 11b/15/16:

```
CELL 0 → CELL 1 → CELL 2 → CELL 3 → CELL 4 → CELL 4A
→ CELL 6 → CELL 7 → CELL 8b → CELL 10 → CELL 10b
```

Cell 10b takes ~5 minutes and writes `scalar_offset_915mhz.json`.
Then open `sionna2_915mhz_dem_simulation.ipynb` → run CELL 4A →
confirm it prints: `[Cell 4A] Scalar offset loaded: X.XX dB`

---

### Transfer to Sionna 2 DEM

After Cell 10b and Cell 11b complete, two JSON files are written:

| File | Written by | Loaded by |
|------|-----------|----------|
| `scalar_offset_915mhz.json` | Cell 10b | Sionna 2 DEM Cell 4A → applied in CELL 8 |
| `calibrated_materials_915mhz.json` | Cell 11b | Sionna 2 DEM Cell 4A → replaces ITU defaults |

Both files are auto-loaded with fallback to defaults if not found.

---

### Notebook Structure

| Cell | Name | Purpose |
|------|------|---------|
| CELL 0 | Environment Setup | Imports, path setup, TF config |
| CELL 1 | Scene Detection | Auto-detect scene XML, read bounds |
| CELL 2 | Global Configuration | Frequency, TX power, ray parameters |
| CELL 3 | Coordinate Utilities | GPS ↔ local XY, DEM elevation, ray-cast Z |
| CELL 4 | Load Scene | Load Mitsuba scene, configure antennas |
| CELL 4A | Material Properties | Assign ITU-R P.2040-2 EM properties, load calibrated JSON |
| CELL 6 | Load Transmitter | GPS → BNG → local XY, ray-cast height AGL |
| CELL 7 | Load Receivers | 1 200 Ofcom measurement points → local XY |
| CELL 8 | Coverage Map | Pre-calibration coverage map (visual check, optional) |
| CELL 8b | Calibration Targets | Load Ofcom RSSI, build calib_receivers |
| CELL 10 | Trainable Materials | Create `tf.Variable` ε_r, σ, S per material |
| CELL 10b | Scalar Offset | Baseline: optimise global dB offset (~5 min) |
| CELL 11b | Material Calibration | Diff-RT: optimise ε_r, σ, S via trace_paths + compute_fields (~2–3 h) |
| CELL 15 | Residual MLP | Physics-informed 50-feature MLP on RT residuals |
| CELL 16 | MaterialMLP | Scene-conditioned MLP → material params → generalises to new scenes |
| CELL 12 | TX Orientation | Optimise TX antenna pointing direction |
| CELL 13 | Post-Calibration | Final coverage map + per-receiver error statistics |

---

### Key Fixes Applied

| Fix | Problem | Solution |
|-----|---------|---------|
| Scene file | `scene_with_roads_019.xml` (7 objects) | Changed to `scene_with_full_019.xml` (11 objects) |
| GPU OOM | 10M rays × 1 200 RX → 13 GB tensor | Batched: 50 RX × 1M rays per batch |
| RMSE = 149 dB | `eps=1e-30` → RSSI=−271 dBm (finite, wrong) | Valid mask: `RSSI > −150 dBm` |
| TX double-count | `paths_to_rssi` added TX power twice | Removed — Sionna 0.19 embeds TX power in `paths.a` |
| Material sync | 0.19 and Sionna 2 had different ε_r/σ values | Both now use ITU-R P.2040-2 (2023) Table 3 |
| Water ε_r = 30 | mat-water → itu_wet_ground | → itu_water (ε=80, σ=0.010) ITU-R P.527 |
| Vegetation ε_r = 5.31 | mat-vegetation → itu_concrete | → itu_vegetation (ε=1.50) ITU-R P.833 |


---
## CELL 0 · Environment Setup & Imports

Imports all required libraries and configures TensorFlow GPU memory growth.  
Run this cell first after every kernel restart.


In [ ]:
import os, sys, json, csv, time, warnings, importlib
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from pyproj import Transformer

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

# ── Mitsuba variant MUST be set before import sionna ─────────────────────────
# Sionna 0.19 registers radio-material plugin with the active variant.
# Calling mi.set_variant() after import sionna resets the plugin registry.
_HAS_MI = False
try:
    import mitsuba as mi
    try:   mi.set_variant('cuda_ad_mono_polarized')
    except: mi.set_variant('llvm_ad_mono_polarized')
    _HAS_MI = True
    print(f'Mitsuba : {mi.variant()}')
except ImportError:
    print('Mitsuba : NOT available – ray-cast ground height disabled')

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

# ── rasterio (DEM lookup) ──────────────────────────────────────────────────────
_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
except ImportError:
    print('rasterio: NOT available – DEM elevation disabled')

# ── OFDM helpers (diff-rt NMSE loss) ──────────────────────────────────────────
_HAS_OFDM = False
for _pkg in ('sionna.channel', 'sionna.channel.ofdm'):
    try:
        _m = importlib.import_module(_pkg)
        cir_to_ofdm_channel    = _m.cir_to_ofdm_channel
        subcarrier_frequencies = _m.subcarrier_frequencies
        _HAS_OFDM = True
        print(f'OFDM    : OK  ({_pkg})')
        break
    except (ImportError, AttributeError):
        continue
if not _HAS_OFDM:
    print('OFDM    : NOT found – power-domain fallback will be used')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')
print(f'GPU(s)  : {[g.name for g in tf.config.list_physical_devices("GPU")]}')

# ── Shared helpers ─────────────────────────────────────────────────────────────
def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

---
## CELL 1 · Project & Scene Detection

Auto-detects `scene_with_full_019.xml` from the project folder.  
Reads scene bounds (`WEST`, `EAST`, `SOUTH`, `NORTH`) and scene centre from XML `<default>` tags.  
Sets `XML_OK = True` if the scene file is found — Cell 4 will raise an error if `XML_OK` is False.


In [ ]:
# ── Nottingham DEM scene (EA LiDAR 1m DTM, 915 MHz Ofcom 2018) ───────────────
CITY_NAME    = 'Nottingham'
BASE_DIR     = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', 'nottingham_ofcom2018_915mhz_dem'))
PROJECT_PATH = BASE_DIR
SCENE_XML    = os.path.join(BASE_DIR, 'scene_v3_enhanced', 'scene_with_full_019.xml')
# DEM path — try common locations in order
DEM_TIFF = next((p for p in [
    os.path.join(BASE_DIR, 'dem.tif'),
    os.path.join(BASE_DIR, 'scene', 'dem.tif'),
    os.path.join(BASE_DIR, 'scene', 'dem_wgs84.tif'),
] if os.path.exists(p)), os.path.join(BASE_DIR, 'dem.tif'))

# Nottingham DEM scene bbox (must match sionna2_915mhz_dem_simulation CELL 1)
WEST, EAST   = -1.267685, -1.119832
SOUTH, NORTH =  52.943165, 53.003037

XML_OK = os.path.exists(SCENE_XML)
print(f'Project   : {CITY_NAME} — DEM + Roads 915 MHz')
print(f'Base dir  : {BASE_DIR}')
print(f'Scene XML : {SCENE_XML}  {"✓" if XML_OK else "✗ NOT FOUND"}')
print(f'DEM       : {DEM_TIFF}  {"✓" if os.path.exists(DEM_TIFF) else "✗ NOT FOUND"}')
print(f'Bbox      : lon [{WEST}, {EAST}]  lat [{SOUTH}, {NORTH}]')

# Read scene XML to confirm version
if XML_OK:
    try:
        import xml.etree.ElementTree as _ET
        _root = _ET.parse(SCENE_XML).getroot()
        _ver  = _root.get('version', 'unknown')
        _maj  = int(_ver.split('.')[0]) if _ver and _ver[0].isdigit() else 0
        _ok_ver = _ver.startswith('2.') or _ver.startswith('3.')
        print(f'XML version : {_ver}  {"✓ Mitsuba 2.x (Sionna 0.19)" if _ver.startswith("2.") else ("✓ Mitsuba 3.x" if _ver.startswith("3.") else "✗ unknown version")}')
    except Exception as _e:
        print(f'XML parse warning: {_e}')

# Output directory
OUTPUT_DIR  = os.path.join(BASE_DIR, 'results', 'diff_rt')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

# ── Sionna 2 DEM working directory (transfer target) ─────────────────────────
# scalar_offset_915mhz.json is written here so Sionna 2 DEM Cell 4A can load it
DEM_BASE_DIR = BASE_DIR   # same project root — change if DEM notebook uses different path
print(f'DEM dir    : {DEM_BASE_DIR}')

---
## CELL 2 · Global Configuration

All simulation parameters in one place. Edit here before running the calibration.

| Parameter | Variable | Value | Notes |
|-----------|----------|-------|-------|
| TX conducted power | `TX_CONDUCTED_DBM` | 49.0 dBm | From Ofcom site record |
| RX extra gain | `RX_EXTRA_GAIN_DB` | 0.0 dB | No artificial offsets |
| Max ray depth | `CALIB_DEPTH` | 5 | Reflections per path |
| Rays per batch | `NUM_SAMPLES_PS` | 1 000 000 | Reduced from 10M to prevent GPU OOM |
| Calibration steps | `CALIB_STEPS` | 5 000 | Scalar offset baseline |
| Calibration receivers | `CALIB_N_RX` | 1 200 | All Ofcom measurement points |


In [ ]:
import tensorflow as _tf_gpu
_gpus = _tf_gpu.config.list_physical_devices('GPU')
if _gpus:
    for _g in _gpus:
        _tf_gpu.config.experimental.set_memory_growth(_g, True)
    print(f"GPU memory growth enabled for {len(_gpus)} GPU(s)")

# Use all 12 CPU cores for TF inter/intra-op parallelism
_tf_gpu.config.threading.set_inter_op_parallelism_threads(12)
_tf_gpu.config.threading.set_intra_op_parallelism_threads(12)
print("TF threading: 12 inter-op + 12 intra-op threads")

# ── Ofcom site parameters (Nottingham 915 MHz DEM run) ───────────────────────
# Matches sionna2_915mhz_dem_simulation CELL 1 exactly — no offsets.
# Formula: RSSI = TX_CONDUCTED_DBM + 10*log10(sum|a|^2) + RX_EXTRA_GAIN_DB
FREQUENCY_HZ     = 915.95e6     # Ofcom 2018 drive-test frequency
TX_HEIGHT_M      = 17.0         # TX antenna height AGL (m)
TX_CONDUCTED_DBM = 49.0         # dBm (conducted power at antenna port)
TX_GAIN_DBI      = 1.3          # dBi collinear omni — handled by Sionna pattern

# RX chain — no corrections applied (matches DEM simulation notebook)
RX_AGL_M         = 1.5
RX_EXTRA_GAIN_DB = 0.0          # no chain gain/loss correction
SYS_GAIN         = 0.0
SITE_CORRECTION_DB = 0.0

# TX GPS position (Ofcom 2018 Nottingham site)
TX_LAT           = 52.9863
TX_LON           = -1.2559

BANDWIDTH_HZ    = 20e6
NOISE_FLOOR     = -109.0    # Ofcom spec: system noise floor (dBm)

# ── Coordinate system ─────────────────────────────────────────────────────────
UTM_EPSG        = 32630   # UTM zone 30N — covers UK/Nottingham

# ── Input / output CSVs ───────────────────────────────────────────────────────
RX_CSV          = os.path.join(BASE_DIR, 'scene', 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(BASE_DIR, 'scene', 'measurements_with_pathloss.csv')
TX_CSV          = os.path.join(BASE_DIR, 'scene', 'transmitter_positions.csv')

# ── OFDM parameters ───────────────────────────────────────────────────────────
NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

# ── Path solver parameters ────────────────────────────────────────────────────
MAX_DEPTH      = 15
NUM_SAMPLES_CM = 1_000_000     # minimal — coverage map is visualisation only
NUM_SAMPLES_PS = 2_000_000   # 20M — safe ceiling for Sionna 0.19 TF compute_paths() (near-field ≤400m)
GRID_SIZE_M    = 5.0

# ── Differentiable RT calibration ─────────────────────────────────────────────
CALIB_STEPS    = 500      # official paper: 10000; 500 is practical for RSSI-only
CALIB_LR       = 5e-3     # Adam learning rate
CALIB_N_RX     = 1200     # all 1200 Ofcom receivers (bad-PL outliers filtered)
CALIB_BATCH    = 5       # receivers per compute_paths() batch — balance GPU memory vs speed
CALIB_NUM_SAMP = 500_000  # num_samples for compute_paths() per step
CALIB_DEPTH    = 5        # NVLabs official depth (3-5)

# ── TX orientation optimization ───────────────────────────────────────────────
ORI_STEPS    = 50
ORI_LR       = 0.01
ORI_NUM_SAMP = 1_000_000

_tx_w    = 10**((TX_CONDUCTED_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

print('=' * 65)
print('SYSTEM CONFIGURATION — Nottingham 915 MHz DEM (Ofcom 2018)')
print('=' * 65)
print(f'Frequency   : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX conducted: {TX_CONDUCTED_DBM} dBm  (pos: {TX_LAT}, {TX_LON}, h={TX_HEIGHT_M}m)')
print(f'SYS_GAIN    : {SYS_GAIN:.1f} dB  (RX extra gain: {RX_EXTRA_GAIN_DB} dB)')
print(f'SNR scale   : {SNR_SCALE:.2e}')
print(f'UTM EPSG    : {UTM_EPSG}')
print(f'Max depth   : {MAX_DEPTH}')
print(f'RX CSV      : {RX_CSV}  {"✓" if os.path.exists(RX_CSV) else "✗  (run CELL 6c in main notebook first)"}')
print(f'Meas CSV    : {MEASUREMENT_CSV}  {"✓" if os.path.exists(MEASUREMENT_CSV) else "✗"}')
print(f'Calib       : {CALIB_STEPS} steps  LR={CALIB_LR}  N={CALIB_N_RX}  batch={CALIB_BATCH}')
print('=' * 65)

---
## CELL 3 · Coordinate Utilities + DEM Elevation

Helper functions for coordinate conversion and height lookup:

| Function | Input | Output |
|----------|-------|--------|
| `gps_to_local(lon, lat)` | WGS84 GPS | Local XY (m) relative to scene origin |
| `local_to_gps(x, y)` | Local XY | WGS84 GPS |
| `get_dem_elevation(x, y)` | Local XY | Absolute height (m ASL) from DTM raster |
| `ray_cast_ground_z(x, y)` | Local XY | Terrain height via Mitsuba ray intersection |

**Coordinate system:** WGS84 GPS → UTM EPSG:32630 (UK zone 30N) → subtract scene origin → Sionna local XY.


In [ ]:
# Derive scene center from bbox
center_lon = (WEST + EAST)   / 2
center_lat = (SOUTH + NORTH) / 2

gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    """GPS (lon, lat) to Sionna local XY (metres from scene origin)."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Sionna local XY to GPS (lon, lat)."""
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

# -- DEM bilinear lookup -----------------------------------------------------
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())
if _is_bng_dem:
    utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)

# ── PLY-based terrain interpolator (same method as Sionna 2 DEM) ────────────
# Reads scene_v3_enhanced/meshes/terrain.ply — same mesh Sionna ray-traces.
# Guarantees RX/TX Z is consistent with scene geometry.
_terrain_ply_path = os.path.join(BASE_DIR, 'scene_v3_enhanced', 'meshes', 'terrain.ply')
_ply_interp_lin = _ply_interp_near = None

def _load_ply_verts(path):
    import struct
    with open(path, 'rb') as _f:
        hdr = []
        while True:
            line = _f.readline().decode('ascii', errors='ignore').strip()
            hdr.append(line)
            if line == 'end_header': break
        nv = next(int(l.split()[-1]) for l in hdr if l.startswith('element vertex'))
        is_bin = any('binary' in l for l in hdr)
        if is_bin:
            raw = _f.read()
            return np.frombuffer(raw[:nv*12], dtype=np.float32).reshape(-1, 3).copy()
        return np.array([list(map(float, _f.readline().split()[:3])) for _ in range(nv)],
                        dtype=np.float32)

if os.path.exists(_terrain_ply_path):
    from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
    _tv = _load_ply_verts(_terrain_ply_path)
    _ply_interp_lin  = LinearNDInterpolator(_tv[:, :2], _tv[:, 2])
    _ply_interp_near = NearestNDInterpolator(_tv[:, :2], _tv[:, 2])
    print(f'Terrain PLY : {os.path.basename(_terrain_ply_path)}  '
          f'verts={len(_tv):,}  z=[{_tv[:,2].min():.1f}, {_tv[:,2].max():.1f}] m')
else:
    print(f'Terrain PLY : not found at {_terrain_ply_path} — falling back to dem.tif')

def _ply_terrain_z(local_x, local_y):
    if _ply_interp_lin is None: return None
    v = _ply_interp_lin(float(local_x), float(local_y))
    if v is None or np.isnan(v):
        v = _ply_interp_near(float(local_x), float(local_y))
    return float(v)

def get_dem_elevation(local_x, local_y):
    # PLY primary: same mesh as scene geometry → guaranteed Z consistency
    ply_z = _ply_terrain_z(local_x, local_y)
    if ply_z is not None:
        return ply_z
    # dem.tif fallback
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    if _is_bng_dem:
        px, py = utm_to_bng.transform(utm_x, utm_y)
    else:
        px, py = utm_to_gps.transform(utm_x, utm_y)  # WGS84 lon/lat
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if not _HAS_MI: return get_dem_elevation(x, y)
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

def _scene_bbox():
    """Return (xmin, xmax, ymin, ymax) in local metres. Works Sionna 0.19 and 2.0."""
    for _attr in ('mi_scene', '_scene'):
        try:
            bb = getattr(scene, _attr).bbox()
            return float(bb.min[0]), float(bb.max[0]), float(bb.min[1]), float(bb.max[1])
        except Exception:
            continue
    # Fallback: compute from lon/lat bbox
    wx, sy = gps_to_utm.transform(WEST,  SOUTH)
    ex, ny = gps_to_utm.transform(EAST,  NORTH)
    return (wx - utm_center_x, ex - utm_center_x,
            sy - utm_center_y, ny - utm_center_y)

print('Coordinate utilities ready.')
print(f'  center: ({center_lon:.4f}, {center_lat:.4f})')

---
## CELL 4 · Load 3-D Scene & Configure Antennas

Loads `scene_with_full_019.xml` — the Sionna 0.19 / Mitsuba scene with 11 geometry objects and 17 ITU-R materials.

**Scene contents:**

| Category | Objects | Materials |
|----------|---------|-----------|
| Buildings | 7 PLY files (brick, concrete, glass, metal, wood variants) | itu_brick, itu_concrete, itu_glass, itu_metal, itu_wood |
| Terrain | terrain.ply (EA LiDAR 1m DTM) | itu_wet_ground |
| Roads | road_itu_asphalt.ply | itu_concrete (asphalt proxy) |
| Water | water.ply (River Trent + Canal) | itu_water (ε=80, σ=0.010) |
| Vegetation | vegetation.ply (parks, gardens) | itu_vegetation (ε=1.50, σ=0.0) |

**Fix applied:** `merge_shapes=True` removed — not supported in Sionna 0.19 `load_scene()`.


In [ ]:
if not XML_OK:
    raise RuntimeError(
        'scene.xml not found. Complete Steps 1-3 in the sionna_web UI:\n'
        '  1. Draw area on map\n'
        '  2. Configure materials\n'
        '  3. Click "Generate 3-D Scene"')

print(f'Loading scene from {SCENE_XML} ...')
try:
    scene = load_scene(SCENE_XML, merge_shapes=False)
except TypeError:
    scene = load_scene(SCENE_XML)   # Sionna 0.19 fallback — no merge_shapes param
scene.frequency = FREQUENCY_HZ

def _make_array(cfg):
    return PlanarArray(
        num_rows           = cfg.get('num_rows',           1),
        num_cols           = cfg.get('num_cols',           1),
        vertical_spacing   = cfg.get('vertical_spacing',   0.5),
        horizontal_spacing = cfg.get('horizontal_spacing', 0.5),
        pattern            = cfg.get('pattern',            'iso'),
        polarization       = cfg.get('polarization',       'V'),
    )

# Antenna pattern: 'dipole' = half-wave dipole donut (~2.15 dBi, null at zenith/nadir)
# Matches DEM simulation: ANTENNA_PATTERN='donut' -> pattern='dipole'
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='dipole',
                             polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='dipole',
                             polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')
print(f'TX array      : {scene.tx_array}')

# Scene bounding box (uses _scene_bbox() defined in CELL 3)
try:
    _xmin, _xmax, _ymin, _ymax = _scene_bbox()
    print(f'Scene bbox    : X=[{_xmin:.0f}, {_xmax:.0f}]  Y=[{_ymin:.0f}, {_ymax:.0f}]')
except Exception as _be:
    print(f'Scene bbox    : {_be}')


---
## CELL 4A · Assign ITU-R P.2040-2 Material Properties

Assigns electromagnetic properties to each scene material following ITU-R P.2040-2 (buildings),  
ITU-R P.527 (water), and ITU-R P.833 (vegetation) at 915 MHz.

| Material | ε_r | σ (S/m) | Scatter S | Standard | Notes |
|----------|-----|---------|-----------|----------|-------|
| itu_concrete | 5.24 | 0.130 | 0.40 | P.2040-2 | |
| itu_brick | 3.91 | 0.024 | 0.25 | P.2040-2 | |
| itu_glass | 6.27 | 0.012 | 0.08 | P.2040-2 | |
| itu_wood | 1.99 | 0.005 | 0.30 | P.2040-2 | |
| itu_metal | 1.00 | 1×10⁷ | 0.05 | P.2040-2 | Perfect conductor |
| itu_wet_ground | 30.0 | 0.020 | 0.35 | P.2040-2 | Terrain surface |
| **itu_water** | **80.0** | **0.010** | 0.02 | **P.527** | **Fixed: was itu_wet_ground (ε=30)** |
| **itu_vegetation** | **1.50** | **0.000** | 0.10 | **P.833** | **Fixed: was itu_concrete (ε=5.31)** |


In [ ]:
# ITU-R P.2040-2 (2023) Table 3 — frequency-portable material properties
# Synced with sionna2_915mhz_dem_simulation.ipynb Cell 4A
# Format per entry: (a, b, c, d, s, xpd, wt)
#   εr(f) = a · f^b   σ(f) = c · f^d   (f in GHz)
#   s  = scattering coefficient   xpd = cross-pol disc.   wt = wall thickness (m)
_ITU_P2040 = {
    #                   a       b       c        d       s     xpd   wt(m)
    'concrete':        (5.31,   0,      0.0326,  0.8095, 0.20, 0.10, 0.30),
    'brick':           (3.91,   0,      0.0238,  0,      0.25, 0.15, 0.23),
    'glass':           (6.27,   0,      0.0043,  1.1925, 0.08, 0.02, 0.012),
    'metal':           (1.00,   0,      1e7,     0,      0.05, 0.01, 0.005),
    'wood':            (1.99,   0,      0.0047,  1.0718, 0.15, 0.10, 0.05),
    'plasterboard':    (2.73,   0,      0.0085,  0.9395, 0.10, 0.05, 0.02),
    'marble':          (7.07,   0,      0.0200,  0,      0.05, 0.08, 0.03),
    'asphalt':         (2.56,   0,      0.0050,  0,      0.30, 0.15, 0.05),
    'vegetation':      (1.50,   0,      0.0020,  0.50,   0.60, 0.50, 0.10),
    'water':           (80.0,   0,      0.0100,  0,      0.03, 0.02, 0),
    'wet_ground':      (30.0,  -0.4,    0.1500,  1.30,   0.35, 0.25, 0),
    'medium_ground':   (15.0,  -0.1,    0.0350,  1.63,   0.30, 0.25, 0),
    'very_dry_ground': ( 3.0,   0,      0.00015, 2.52,   0.20, 0.20, 0),
    'plywood':         (2.71,   0,      0.0140,  0,      0.15, 0.10, 0.02),
    'chipboard':       (2.58,   0,      0.0120,  0,      0.14, 0.10, 0.02),
    'ceiling_board':   (1.50,   0,      0.0060,  0,      0.13, 0.05, 0.02),
    'floorboard':      (2.00,   0,      0.0100,  0,      0.16, 0.10, 0.02),
}
_DEFAULT_MAT = (4.0, 0, 0.08, 0, 0.20, 0.10, 0.10)  # fallback 7-tuple

_freq_ghz = FREQUENCY_HZ / 1e9

def _itu_at_freq(key, f_ghz):
    """Return (er, sigma, s, xpd, wt) at frequency f_ghz using P.2040-2 power law."""
    a, b, c, d, s, xpd, wt = _ITU_P2040.get(key, _DEFAULT_MAT)
    return float(a * (f_ghz ** b)), float(c * (f_ghz ** d)), float(s), float(xpd), float(wt)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_P2040:
        if key in n: return key
    for key in _ITU_P2040:
        if any(part in n for part in key.split('_')): return key
    return None

print('=' * 70)
print('ASSIGNING ITU-R P.2040-2 MATERIAL PROPERTIES  (auto-match by name)')
print(f'  Frequency : {_freq_ghz:.4f} GHz')
print('=' * 70)
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    er, sigma, s, xpd, wt = _itu_at_freq(key, _freq_ghz) if key else (
        _DEFAULT_MAT[0], _DEFAULT_MAT[2], _DEFAULT_MAT[4], _DEFAULT_MAT[5], _DEFAULT_MAT[6])
    try: mat.relative_permittivity = er
    except Exception: pass
    try: mat.conductivity = sigma
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, s); break
            except: pass
    for a_ in ('xpd_coefficient', 'xpd_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, xpd); break
            except: pass
    try: mat.thickness = wt
    except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  '
          f'er={er:.2f}  σ={sigma:.4g}  S={s:.2f}  xpd={xpd:.2f}  wt={wt:.3f}m')
print('Done.')

---
## CELL 6 · Load Transmitter

Reads TX position from `transmitter_positions.csv`.  
Pipeline: GPS (lon, lat) → UTM EPSG:32630 → subtract scene origin → Sionna local XY.  
Height: ray-cast against terrain mesh to get ground Z, then add antenna height AGL.  
TX created with `power_dbm = TX_CONDUCTED_DBM = 49.0` — Sionna 0.19 embeds this into `paths.a`.


In [ ]:
print('=' * 70)
print('CELL 6 – LOAD TRANSMITTER')
print('=' * 70)

for nm in list(scene.transmitters.keys()):
    scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', TX_CONDUCTED_DBM))
    source   = 'CSV'
else:
    # Fallback: use hardcoded Ofcom TX parameters (matches DEM simulation)
    tx_name  = 'tx0'
    tx_lon   = TX_LON
    tx_lat   = TX_LAT
    tx_agl   = TX_HEIGHT_M
    tx_power = TX_CONDUCTED_DBM
    source   = 'hardcoded (TX_LON/TX_LAT/TX_HEIGHT_M)'
    print(f'  TX CSV not found – using hardcoded Ofcom TX parameters')

print(f'[1] Source      : {source}')
print(f'    GPS         : ({tx_lon:.6f}, {tx_lat:.6f})  AGL={tx_agl:.1f} m')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
print(f'[2] Local XY    : ({local_x:.2f}, {local_y:.2f})')

# DEM (dem.tif) gives terrain height in scene Z coordinates — primary source
# ray_cast_ground_z as fallback only; never use bbox.min[2] (scene geometric floor)
ground_z = get_dem_elevation(local_x, local_y)
if ground_z == 0.0:
    ground_z = ray_cast_ground_z(local_x, local_y)
abs_z = ground_z + tx_agl
print(f'[3] Ground Z    : {ground_z:.2f} m (DEM)  +  AGL {tx_agl:.1f} m  →  abs Z={abs_z:.2f} m')

tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)),
                 power_dbm=float(tx_power))
scene.add(tx)
print(f'[4] ✓ Added TX "{tx_name}"  pos=({local_x:.1f}, {local_y:.1f}, {abs_z:.1f})  conducted={tx_power:.1f} dBm  antenna=dipole  EIRP={tx_power + 2.15:.1f} dBm')

---
## CELL 7 · Load Receivers

Reads 1 200 receiver positions from `receiver_locations.csv` and measured RSSI from `measurements_with_pathloss.csv`.  
Same GPS → local XY pipeline as Cell 6.  
All 1 200 receivers are loaded — no distance filter applied.

**Output variables:**
- `receivers` — list of Sionna `Receiver` objects
- `rx_meas_rssi` — measured RSSI (dBm) array, aligned to receivers list


In [ ]:
print('=' * 70)
print('CELL 7 – LOAD RECEIVERS')
print('=' * 70)

for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV).head(1200)  # cap at first 1200 RX
    print(f'[1] Loaded {len(df_rx)} receivers (capped at 1200) from {RX_CSV}')
    print('[2] Converting GPS → local XY + ray-cast ground Z ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon  = float(row['lon'])
        lat  = float(row['lat'])
        agl  = float(row.get('height', RX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        gz   = get_dem_elevation(x, y)         # DEM terrain height — same Z system as scene
        if gz == 0.0:
            gz = ray_cast_ground_z(x, y)   # fallback
        z    = gz + agl
        nm   = str(row.get('name', f'RX_{i+1:04d}'))
        rx   = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx)
        receivers.append(rx)
    print(f'    Done in {time.time()-t0:.2f} s')
else:
    print('  RX CSV not found – using project.json receivers')
    for rx_cfg in _ant.get('receivers', [{'name':'rx0','position':[100,0,RX_AGL_M]}]):
        pos = rx_cfg.get('position', [100, 0, RX_AGL_M])
        nm  = rx_cfg.get('name', f'rx{len(receivers)}')
        rx  = Receiver(name=nm, position=(float(pos[0]), float(pos[1]), float(pos[2])))
        scene.add(rx)
        receivers.append(rx)

print(f'[3] {len(receivers)} receivers placed')
print('[4] First 5 receivers:')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f}, {y:8.1f})  Z={z:.2f}  GPS=({lon:.5f}, {lat:.5f})')

try:
    _bbox = scene.mi_scene.bbox()
    ok_x  = all(float(_bbox.min[0]) <= _safe(r.position[0]) <= float(_bbox.max[0]) for r in receivers)
    ok_y  = all(float(_bbox.min[1]) <= _safe(r.position[1]) <= float(_bbox.max[1]) for r in receivers)
    print(f'[5] All inside scene bbox: X={ok_x}  Y={ok_y}')
except: pass

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

# Load Ofcom measurements for calibration
df_meas = None
rssi_measured_all = None
if os.path.exists(MEASUREMENT_CSV):
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    _rssi_col = next((c for c in df_meas.columns
                      if any(k in c.lower() for k in ['rssi','measurement','dbm','signal'])), None)
    if _rssi_col:
        rssi_measured_all = df_meas[_rssi_col].values.astype(np.float32)
        print(f'Ofcom RSSI loaded : {len(rssi_measured_all)} samples  '
              f'range {rssi_measured_all.min():.1f}\u2013{rssi_measured_all.max():.1f} dBm')
    else:
        print(f'WARNING: no RSSI column in {MEASUREMENT_CSV}')
else:
    print(f'WARNING: {MEASUREMENT_CSV} not found \u2014 run CELL 6c in main notebook first')


---
## CELL 8 · Pre-Calibration Coverage Map

Generates a coverage map using ITU-R default material properties — a visual sanity check  
before calibration begins. Shows predicted RSSI across the scene at 20m resolution.  
This cell is optional; skip if runtime is a concern.


In [ ]:
print('Pre-calibration coverage map (ITU material defaults) ...')

try:
    _bbox = scene.mi_scene.bbox()
    cx = (float(_bbox.min[0]) + float(_bbox.max[0])) / 2
    cy = (float(_bbox.min[1]) + float(_bbox.max[1])) / 2
except Exception:
    cx = cy = 0.0

ground_z_centre = ray_cast_ground_z(cx, cy)
if ground_z_centre == 0.0:
    ground_z_centre = get_dem_elevation(cx, cy)
cm_height = ground_z_centre + RX_AGL_M
print(f'  Coverage map plane Z = {ground_z_centre:.2f} + {RX_AGL_M} = {cm_height:.2f} m')

try:
    cm_pre = scene.coverage_map(
        cm_cell_size        = [GRID_SIZE_M, GRID_SIZE_M],
        max_depth           = MAX_DEPTH,
        num_samples         = NUM_SAMPLES_CM,
        los                 = True,
        specular_reflection = True,
        diffuse_reflection  = True,
        refraction          = True,
        diffraction         = True,
    )
except TypeError:
    # Sionna 0.19 uses different parameter names
    cm_pre = scene.coverage_map(
        cm_cell_size = [GRID_SIZE_M, GRID_SIZE_M],
        max_depth    = MAX_DEPTH,
        num_samples  = NUM_SAMPLES_CM,
    )
cm_pre_np = _cm_to_numpy(cm_pre)

pg_db = 10 * np.log10(cm_pre_np[0] + 1e-30)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pg_db, origin='lower', cmap='jet',
               vmin=np.nanpercentile(pg_db, 5), vmax=np.nanpercentile(pg_db, 99))
plt.colorbar(im, ax=ax, label='Path Gain (dB)')
ax.set_title('Pre-Calibration Coverage Map – ITU Material Defaults')
ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_pre_calibration.png'), dpi=150)
plt.show()

# ── Interpolate to RX positions via KDTree ────────────────────────────────────
H, W = pg_db.shape
try:
    _bbox = scene.mi_scene.bbox()
    _gx_min, _gx_max = float(_bbox.min[0]), float(_bbox.max[0])
    _gy_min, _gy_max = float(_bbox.min[1]), float(_bbox.max[1])
except Exception:
    _gx_min = _gy_min = -500.0; _gx_max = _gy_max = 500.0

x_centers = np.linspace(_gx_min, _gx_max, W)
y_centers  = np.linspace(_gy_min, _gy_max, H)
XX, YY = np.meshgrid(x_centers, y_centers)
tree   = KDTree(np.column_stack([XX.ravel(), YY.ravel()]))
rx_coords   = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
_, _indices = tree.query(rx_coords)
pg_at_rx_pre = pg_db.ravel()[_indices]

print(f'Pre-calibration path-gain at RX:  '
      f'mean={np.mean(pg_at_rx_pre):.1f} dB  '
      f'min={np.min(pg_at_rx_pre):.1f} dB  max={np.max(pg_at_rx_pre):.1f} dB')

---
## CELL 8b · Calibration Targets

Builds the calibration dataset:
- `calib_receivers` — list of `Receiver` objects to use in calibration
- `calib_rssi_meas` — matching measured RSSI (dBm) tensor

All 1 200 receivers are included (distance filter removed).  
Measured RSSI loaded from `measurements_with_pathloss.csv` column `RSSI_dBm`.


In [ ]:
# ====================================================================
# CELL 8b — CALIBRATION TARGET
# ====================================================================
# If Ofcom measurements are available: use measured RSSI dBm as target
# (power-domain calibration — correct approach for drive-test CSV data)
# Fallback: self-supervised NMSE using compute_paths() (diff-rt demo mode)
# ====================================================================

CALIB_MODE = 'ofcom'   # 'ofcom' = use measured RSSI  |  'self' = self-supervised NMSE

if CALIB_MODE == 'ofcom' and rssi_measured_all is not None:
    # ── Stratified sample of CALIB_N_RX receivers ──────────────────────────
    import math as _math
    _rx_names = [rx.name for rx in receivers]
    _rx_rssi  = {r: rssi_measured_all[i] for i, r in enumerate(_rx_names)
                 if i < len(rssi_measured_all) and np.isfinite(rssi_measured_all[i])}

    # Compute distances for stratified sampling
    _tx_x, _tx_y = gps_to_local(TX_LON, TX_LAT)[:2]
    _dists = {rx.name: float(np.sqrt(
        (_safe(rx.position[0]) - _tx_x)**2 +
        (_safe(rx.position[1]) - _tx_y)**2)) / 1000.0
        for rx in receivers}

    # Stratified: equal-count bins across distance range
    _valid_rx = [rx for rx in receivers if rx.name in _rx_rssi]
    _valid_rx.sort(key=lambda r: _dists[r.name])
    # No distance filter — use all receivers across full scene extent
    print(f'  Distance filter : disabled — using all {len(_valid_rx)} RX')
    # ── Filter out bad path-loss measurements (outliers) ─────────────────────
    _pl_vals = np.array([TX_CONDUCTED_DBM - _rx_rssi[rx.name] for rx in _valid_rx
                         if rx.name in _rx_rssi])
    _pl_med  = float(np.median(_pl_vals))
    _pl_std  = float(np.std(_pl_vals))
    _pl_lo   = _pl_med - 3.0 * _pl_std   # lower bound
    _pl_hi   = _pl_med + 3.0 * _pl_std   # upper bound
    _valid_rx = [rx for rx in _valid_rx
                 if _pl_lo <= (TX_CONDUCTED_DBM - _rx_rssi[rx.name]) <= _pl_hi]
    print(f'  Bad-PL filter   : kept {len(_valid_rx)} / {len([r for r in receivers if r.name in _rx_rssi])} '
          f'(|PL - median| <= 3σ,  range [{_pl_lo:.1f}, {_pl_hi:.1f}] dB)')


    _n_bins  = max(1, CALIB_N_RX // 20)
    _bin_sz  = max(1, len(_valid_rx) // _n_bins)
    _sel_idx = []
    for _b in range(_n_bins):
        _bin = _valid_rx[_b*_bin_sz : (_b+1)*_bin_sz]
        _k   = max(1, CALIB_N_RX // _n_bins)
        _step = max(1, len(_bin) // _k)
        _sel_idx += [receivers.index(r) for r in _bin[::_step]][:_k]
    _sel_idx = sorted(set(_sel_idx))[:CALIB_N_RX]

    calib_receivers  = [receivers[i] for i in _sel_idx]
    calib_rssi_meas  = tf.constant(
        [_rx_rssi[rx.name] for rx in calib_receivers], dtype=tf.float32)

    h_ref_tf = None  # not used in 'ofcom' mode

    print(f'Calibration mode  : Ofcom RSSI  ({len(calib_receivers)} receivers)')
    print(f'RSSI range        : {float(calib_rssi_meas.numpy().min()):.1f} – '
          f'{float(calib_rssi_meas.numpy().max()):.1f} dBm')
    _d_sel = [_dists[rx.name] for rx in calib_receivers]
    print(f'Distance range    : {min(_d_sel):.2f} – {max(_d_sel):.2f} km')

else:
    # ── Self-supervised fallback ────────────────────────────────────────────
    CALIB_MODE = 'self'
    print('Calibration mode  : self-supervised NMSE (no Ofcom data)')
    print(f'Computing reference channel  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

    def compute_h_freq(sc, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH):
        paths = sc.compute_paths(
            max_depth=depth, num_samples=num_samp,
            los=True, reflection=True, scattering=True, diffraction=True)
        if _HAS_OFDM:
            try:
                a, tau = paths.cir()
                h = cir_to_ofdm_channel(FREQUENCIES, a, tau, normalize=False)
                return tf.squeeze(h)
            except Exception as _e:
                print(f'  cir() fallback: {_e}')
        a_t = paths.a
        if isinstance(a_t, tuple): a_t = tf.complex(a_t[0], a_t[1])
        if a_t.shape[0] == 1: a_t = a_t[0]
        return tf.cast(tf.reduce_sum(tf.abs(a_t)**2,
                        axis=list(range(1, len(a_t.shape)))), tf.float32)

    h_ref    = compute_h_freq(scene, num_samp=NUM_SAMPLES_PS, depth=MAX_DEPTH)
    h_ref_np = _to_numpy(h_ref)
    h_ref_tf = tf.constant(h_ref_np,
                            dtype=tf.complex64 if np.iscomplexobj(h_ref_np) else tf.float32)
    calib_receivers = list(receivers)
    calib_rssi_meas = None
    print(f'Reference shape   : {h_ref_np.shape}  dtype={h_ref_np.dtype}')

print(f'\nCalibration receivers : {len(calib_receivers)}')
print(f'Mode                  : {CALIB_MODE}')


---
## CELL 10 · Differentiable RT — Setup

Prepares the differentiable calibration pipeline. Contains three sub-cells:

| Sub-cell | Contents | Must run? |
|----------|----------|-----------|
| **CELL 10 (this code cell)** | Creates `trainable_mats` with `tf.Variable` ε_r, σ, S — legacy NVLabs pattern | Yes — defines `_match_itu()`, `_ITU_DB`, `_DEFAULT_MAT` |
| **CELL 10 Loss Functions** | Defines `paths_to_rssi()`, `smape_power_loss()`, `mse_dbm_loss()` | Yes — used by both Cell 10b and Cell 11b |
| **CELL 10b Scalar Offset** | Baseline calibration — optimises a single global dB offset | Optional — run for RMSE=5.72 dB baseline |

**RSSI formula (Sionna 0.19):**
```
RSSI_dBm = 10·log₁₀(Σᵢ|aᵢ|²) + 30 + sys_gain_dB
```
TX power is NOT added — Sionna 0.19 embeds `power_dbm` into `paths.a`.  
Reference: Hoydis et al. 2023, §III.

**Loss function — SMAPE on linear power (NVLabs standard):**
```
L = mean( |P_sim − P_meas| / (P_sim + P_meas + ε) )
```


In [ ]:
orig_params    = {}
original_mats  = {}
trainable_mats = {}
_train_suffix  = '_train'

print('Creating trainable RadioMaterial objects ...')
for mat_name, mat in list(scene.radio_materials.items()):
    if mat_name.endswith(_train_suffix): continue

    # Train all ITU materials that appear in the scene XML
    # (is_used unreliable in Sionna 0.19 — train any material matching ITU pattern)
    _itu_prefixes = ('itu_', 'mat-itu_')
    _used = any(mat_name.startswith(p) for p in _itu_prefixes)
    if not _used:
        # Also check if any scene object uses this material
        _used = any(
            getattr(getattr(obj, 'radio_material', None), 'name', '') == mat_name
            for obj in scene.objects.values())
    if not _used: continue

    key  = _match_itu(mat_name)
    _er0, _sg0, _s0, _xpd0, _wt0 = _itu_at_freq(key, _freq_ghz) if key else (_DEFAULT_MAT[0], _DEFAULT_MAT[2], _DEFAULT_MAT[4], _DEFAULT_MAT[5], _DEFAULT_MAT[6])
    eps0 = _er0; sig0 = _sg0; S0 = _s0
    try:
        v = mat.relative_permittivity
        eps0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    try:
        v = mat.conductivity
        sig0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    for a_ in ('scattering_coefficient','scattering_coeff'):
        if hasattr(mat, a_):
            try: S0 = float(getattr(mat,a_).numpy() if hasattr(getattr(mat,a_),'numpy') else getattr(mat,a_)); break
            except: pass

    orig_params[mat_name] = {'eps_r': eps0, 'sigma': sig0, 'S': S0}

    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    # Log-parameterise conductivity for numerical stability (spans 9 orders of magnitude)
    _log_sig0 = float(np.log(max(sig0, 1e-6)))
    kw = dict(
        relative_permittivity = tf.Variable(eps0,     dtype=tf.float32, name=f'{sn}_eps'),
        conductivity          = tf.Variable(_log_sig0, dtype=tf.float32, name=f'{sn}_log_sig'),
        # NOTE: conductivity variable stores LOG(sigma) — exponentiated when assigned to mat
    )
    try:
        new_mat = RadioMaterial(mat_name+_train_suffix,
                                scattering_coefficient=tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S'),
                                **kw)
    except TypeError:
        new_mat = RadioMaterial(mat_name+_train_suffix, **kw)
        try: new_mat.scattering_coefficient = tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S')
        except: pass

    # Assign exp(log_sigma) as actual conductivity
    try:
        new_mat.conductivity = tf.exp(kw['conductivity'])
    except Exception:
        pass

    # Remove stale _train material if already in scene (re-run safety)
    if (mat_name + _train_suffix) in scene.radio_materials:
        try: scene.remove(mat_name + _train_suffix)
        except Exception: pass
    scene.add(new_mat)
    original_mats[mat_name]  = mat
    trainable_mats[mat_name] = new_mat
    print(f'  {mat_name:<30} → {mat_name+_train_suffix}')
    print(f'    eps_r={eps0:.3f}  log_sig={_log_sig0:.4g}  S={S0:.2f}')

print()
n_redir = 0
for obj_name, obj in scene.objects.items():
    rm = getattr(obj, 'radio_material', None)
    if rm is None: continue
    orig_name = rm.name if hasattr(rm,'name') else str(rm)
    if orig_name in trainable_mats:
        try:
            obj.radio_material = orig_name + _train_suffix
            n_redir += 1
        except Exception as e:
            print(f'  WARNING [{obj_name}]: {e}')

print(f'Redirected {n_redir} scene objects to trainable materials.')
print(f'Trainable materials: {list(trainable_mats.keys())}')


In [ ]:
# ====================================================================
# LOSS FUNCTIONS — matching CALIB_MODE
# ====================================================================

def smape_power_loss(rssi_sim_dbm, rssi_meas_dbm):
    """
    SMAPE on linear power — official diff-rt-calibration loss (Hoydis et al. 2023).
    More robust than MSE on dBm: scale-invariant, symmetric.
    """
    P_sim  = tf.pow(10.0, (rssi_sim_dbm  - 30.0) / 10.0)   # dBm → Watts
    P_meas = tf.pow(10.0, (rssi_meas_dbm - 30.0) / 10.0)
    P_sim  = tf.cast(P_sim,  tf.float32)
    P_meas = tf.cast(P_meas, tf.float32)
    return tf.reduce_mean(
        tf.abs(P_sim - P_meas) / tf.stop_gradient(P_sim + P_meas + 1e-30))

def mse_dbm_loss(rssi_sim_dbm, rssi_meas_dbm):
    """MSE on dBm — simpler alternative, biased toward strong signals."""
    err = tf.cast(rssi_sim_dbm, tf.float32) - tf.cast(rssi_meas_dbm, tf.float32)
    return tf.reduce_mean(err ** 2)

def nmse_loss(h_hat, h_ref):
    """NMSE — used in self-supervised mode only."""
    h_hat = tf.cast(h_hat, h_ref.dtype)
    err   = tf.reduce_mean(tf.abs(h_hat - h_ref)**2)
    ref   = tf.reduce_mean(tf.abs(h_ref)**2) + 1e-30
    return err / ref

def paths_to_rssi(paths, sys_gain_db=0.0):
    # Pure NVLabs formula [Hoydis et al. 2023a, arXiv:2303.11103, §III, Eq.5]
    # P_r [W] = sum_n |a_n|^2  (Sionna 0.19 compute_paths embeds P_t in a_n)
    # RSSI_dBm = 10*log10(P_r) + 30
    # Zero-power receivers -> log10(0) = -inf -> excluded by is_finite() downstream
    # No epsilon, no threshold, no artifact.
    a_t = paths.a
    if isinstance(a_t, tuple):
        a_t = tf.complex(a_t[0], a_t[1])
    a_t  = tf.cast(a_t, tf.complex64)
    pwr  = tf.reduce_sum(
               tf.reshape(tf.abs(a_t)**2, [tf.shape(a_t)[0], -1]), axis=1)
    pwr  = tf.squeeze(tf.cast(pwr, tf.float32))
    rssi_dbm = 10.0 * tf.math.log(pwr) / tf.math.log(10.) + 30.0 + sys_gain_db
    return rssi_dbm

def check_mat(mat):
    """Clamp material properties to physical range."""
    try:
        v = mat.relative_permittivity
        if hasattr(v, 'assign'):
            v.assign(tf.clip_by_value(v, 1.0, 50.0))
    except Exception: pass
    try:
        v = mat.conductivity
        if hasattr(v, 'assign'):
            # If log-parameterised: keep log_sigma in reasonable range
            v.assign(tf.clip_by_value(v, tf.math.log(1e-6), tf.math.log(1e7)))
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try:
                v = getattr(mat, a_)
                if hasattr(v, 'assign'):
                    v.assign(tf.clip_by_value(v, 1e-3, 1.0 - 1e-3))
                break
            except Exception: pass

print(f'Loss functions ready — active mode: {CALIB_MODE}')
print(f'  Ofcom mode : SMAPE on linear power (Hoydis et al. 2023)')
print(f'  Self mode  : NMSE on OFDM channel')


---
## CELL 10b · Scalar Offset Calibration — Baseline

**Purpose:** Establish a baseline RMSE before material parameter tuning.  
Follows the NVLabs *"ITU Materials"* baseline from Hoydis et al. 2023.

**Method:**
1. Pre-trace all paths once with `compute_paths()` — fast, no gradient needed
2. Cache the RSSI values from `paths.a` (fixed, no ray tracing in the loop)
3. Optimise a single scalar `scaling_factor_db` with Adam to minimise SMAPE

**Why scalar offset?**  
A global dB shift aligns the overall power scale. It corrects systematic errors  
(e.g. antenna gain uncertainty) but cannot fix spatially varying multipath errors.  
This is the ceiling for a single-parameter model.

| Parameter | Value |
|-----------|-------|
| Variable | `scaling_factor_db` — global dB shift |
| Optimiser | Adam, LR = 0.5 |
| Steps | `CALIB_STEPS` = 5 000 |
| Loss | SMAPE on linear power |
| Pre-trace | `compute_paths()`, 50 RX/batch, 1M rays/batch |
| Valid mask | RSSI > −150 dBm (excludes zero-power receivers) |

**Expected result:** `scaling_factor_db` ≈ −1.4 dB, RMSE ≈ 5.7 dB


In [ ]:
# ====================================================================
# CELL 10b — NVLabs-style calibration: pre-trace once, then optimise
# ====================================================================
# Step 1: compute_paths() runs ONCE for all calib receivers (offline)
# Step 2: training loop optimises scaling_factor_db — no ray tracing
#          inside the tape → gradient always non-zero (matches NVLabs
#          "ITU Materials" baseline from Hoydis et al. 2023)
# ====================================================================
import random, time

# ── Step 1: pre-trace in batches (avoids GPU OOM) ───────────────────────────
_BATCH_SIZE  = CALIB_BATCH    # receivers per batch (set in config cell)
_NUM_SAMPLES = NUM_SAMPLES_PS # rays per batch    (set in config cell)
print(f'Pre-tracing paths ({len(calib_receivers)} receivers, batch={_BATCH_SIZE}, samples={_NUM_SAMPLES:,}) ...')
print(f'  depth={CALIB_DEPTH}  batches={int(np.ceil(len(calib_receivers)/_BATCH_SIZE))}')

# Detect compute_paths() parameter names once — Sionna 0.19 vs 2.0
import inspect as _insp
_cp_params = set(_insp.signature(scene.compute_paths).parameters.keys())
_USE_OLD_API = 'reflection' in _cp_params      # Sionna 0.19
_USE_NEW_API = 'specular_reflection' in _cp_params  # Sionna 2.0
print(f'compute_paths API: {"Sionna 0.19" if _USE_OLD_API else "Sionna 2.0"}  params={sorted(_cp_params)}')

# Build base config with only parameters that exist in this version
_ps_cfg = {}
if 'max_depth'   in _cp_params: _ps_cfg['max_depth']   = CALIB_DEPTH
if 'num_samples' in _cp_params: _ps_cfg['num_samples'] = _NUM_SAMPLES
if 'los'         in _cp_params: _ps_cfg['los']         = True
if 'diffraction' in _cp_params: _ps_cfg['diffraction'] = True

def _compute_paths_compat():
    if _USE_OLD_API:
        return scene.compute_paths(reflection=True, scattering=False, **_ps_cfg)
    else:
        return scene.compute_paths(specular_reflection=True,
                                   diffuse_reflection=False, **_ps_cfg)

_rssi_batches = []
for _b0 in range(0, len(calib_receivers), _BATCH_SIZE):
    _batch = calib_receivers[_b0 : _b0 + _BATCH_SIZE]
    for nm in list(scene.receivers.keys()):
        scene.remove(nm)
    for rx in _batch:
        scene.add(rx)
    _paths_b = _compute_paths_compat()
    _rssi_b = paths_to_rssi(_paths_b, RX_EXTRA_GAIN_DB)
    _rssi_b = tf.reshape(_rssi_b, [-1])
    _rssi_batches.append(_rssi_b.numpy())
    if (_b0 // _BATCH_SIZE) % 5 == 0:
        print(f'  batch {_b0//_BATCH_SIZE+1}/{int(np.ceil(len(calib_receivers)/_BATCH_SIZE))}  '
              f'solved={int(np.sum(np.isfinite(_rssi_b.numpy())))}/{len(_batch)}')

rssi_sim_cached = tf.constant(np.concatenate(_rssi_batches), dtype=tf.float32)
n_solved = int(tf.reduce_sum(tf.cast(tf.math.is_finite(rssi_sim_cached), tf.int32)).numpy())
print(f'Paths solved : {n_solved}/{len(calib_receivers)} receivers')
print(f'RSSI_sim     : {float(tf.reduce_min(rssi_sim_cached).numpy()):.1f} – {float(tf.reduce_max(rssi_sim_cached).numpy()):.1f} dBm')

# ── Save per-RX path solver results ─────────────────────────────────────────
_rx_csv_rows = []
for _i, rx in enumerate(calib_receivers):
    _rssi_s = float(rssi_sim_cached[_i].numpy()) if _i < len(rssi_sim_cached) else float('nan')
    _rssi_m = float(calib_rssi_meas[_i].numpy()) if _i < len(calib_rssi_meas) else float('nan')
    _valid  = np.isfinite(_rssi_s)
    _pl_s   = TX_CONDUCTED_DBM - _rssi_s if _valid else float('nan')
    _pl_m   = TX_CONDUCTED_DBM - _rssi_m if np.isfinite(_rssi_m) else float('nan')
    _rx_csv_rows.append({
        'rx_name'   : rx.name,
        'x_m'       : float(_safe(rx.position[0])),
        'y_m'       : float(_safe(rx.position[1])),
        'z_m'       : float(_safe(rx.position[2])),
        'rssi_sim_dbm'  : round(_rssi_s, 4),
        'rssi_meas_dbm' : round(_rssi_m, 4),
        'pl_sim_db'     : round(_pl_s, 4) if _valid else float('nan'),
        'pl_meas_db'    : round(_pl_m, 4),
        'paths_found'   : bool(_valid),
    })
import pandas as _pd10
_rx_csv_path = os.path.join(OUTPUT_DIR, 'path_solver_results.csv')
_pd10.DataFrame(_rx_csv_rows).to_csv(_rx_csv_path, index=False)
print(f'Path solver CSV saved → {_rx_csv_path}')

# ── Align calib_rssi_meas to rssi_sim_cached length ──────────────────────────
_n_common  = min(len(rssi_sim_cached), len(calib_rssi_meas))
_meas_trim = calib_rssi_meas[:_n_common]
_sim_trim  = rssi_sim_cached[:_n_common]

# Pure NVLabs filter [Hoy23a §III]: zero-power receivers → log10(0)=-inf
# is_finite() excludes them cleanly — no threshold, no artifact
_valid_mask       = tf.math.is_finite(_sim_trim)
rssi_sim_valid    = tf.boolean_mask(_sim_trim,  _valid_mask)
rssi_meas_valid   = tf.boolean_mask(_meas_trim, _valid_mask)
print(f'Valid pairs  : {int(rssi_sim_valid.shape[0])} (RSSI > -150 dBm threshold)')
# Convert to path loss using TX power from config (no hardcoded values)
pl_sim_valid  = TX_CONDUCTED_DBM - rssi_sim_valid
pl_meas_valid = TX_CONDUCTED_DBM - rssi_meas_valid
if int(rssi_sim_valid.shape[0]) > 0:
    print(f'PL_sim       : {float(tf.reduce_min(pl_sim_valid).numpy()):.1f} – {float(tf.reduce_max(pl_sim_valid).numpy()):.1f} dB')
    print(f'PL_meas      : {float(tf.reduce_min(pl_meas_valid).numpy()):.1f} – {float(tf.reduce_max(pl_meas_valid).numpy()):.1f} dB')

# ── Step 2: optimise scaling_factor_db ───────────────────────────────────────
# scaling_factor_db: global dB shift that aligns sim power to measurements
# Gradient: d(SMAPE)/d(sf) is always non-zero → zero_grads = 0/1
scaling_factor_db = tf.Variable(0.0, dtype=tf.float32, trainable=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.5)   # large LR ok for scalar

# ── Initial RMSE before training ─────────────────────────────────────────────
_rmse_init = float(tf.sqrt(tf.reduce_mean(((pl_sim_valid - pl_meas_valid)**2))).numpy())
_mae_init  = float(tf.reduce_mean(tf.abs(pl_sim_valid - pl_meas_valid)).numpy())
print(f'Before calibration : PL RMSE={_rmse_init:.2f} dB   MAE={_mae_init:.2f} dB  (N={int(rssi_sim_valid.shape[0])})')

history = {'step': [], 'loss': [], 'sf_db': [], 'pl_rmse_db': [], 'mae_db': []}
t0 = time.time()

print(f'\nTraining scaling_factor  ({CALIB_STEPS} steps, LR=0.5)')
print('-' * 55)

for step in range(CALIB_STEPS):
    with tf.GradientTape() as tape:
        rssi_scaled = rssi_sim_valid + scaling_factor_db
        loss = smape_power_loss(rssi_scaled, rssi_meas_valid)

    grads = tape.gradient(loss, [scaling_factor_db])
    optimizer.apply_gradients(zip(grads, [scaling_factor_db]))

    lv  = float(loss.numpy()) * 100
    sfv = float(scaling_factor_db.numpy())
    _pl_sim_step = TX_CONDUCTED_DBM - (rssi_sim_valid + scaling_factor_db)
    _rmse_step   = float(tf.sqrt(tf.reduce_mean((_pl_sim_step - pl_meas_valid)**2)).numpy())
    history['step'].append(step)
    history['loss'].append(lv)
    history['sf_db'].append(sfv)
    history['pl_rmse_db'].append(_rmse_step)
    history['mae_db'].append(float(tf.reduce_mean(tf.abs(_pl_sim_step - pl_meas_valid)).numpy()))

    if step % 50 == 0 or step == CALIB_STEPS - 1:
        n_z = sum(1 for g in grads
                  if g is None or float(tf.reduce_sum(tf.abs(g))) == 0)
        _pl_sim_cal = TX_CONDUCTED_DBM - (rssi_sim_valid + scaling_factor_db)
        _rmse_v = float(tf.sqrt(tf.reduce_mean((_pl_sim_cal - pl_meas_valid)**2)).numpy())
        print(f'  step {step:4d}  PL RMSE={_rmse_v:5.2f} dB  SMAPE×100={lv:+7.2f}  sf={sfv:+.2f} dB  t={time.time()-t0:.0f}s')

print('-' * 55)
print(f'Done in {time.time()-t0:.1f}s')

# ── Save scalar offset optimizer history ──────────────────────────────────────
_hist_csv_path = os.path.join(OUTPUT_DIR, 'scalar_offset_history.csv')
_pd10.DataFrame(history).to_csv(_hist_csv_path, index=False)
print(f'Optimizer history CSV saved → {_hist_csv_path}')

_rssi_cal   = rssi_sim_valid + float(scaling_factor_db.numpy())
_pl_cal     = TX_CONDUCTED_DBM - _rssi_cal
_rmse_final = float(tf.sqrt(tf.reduce_mean((_pl_cal - pl_meas_valid)**2)).numpy())
_mae_final  = float(tf.reduce_mean(tf.abs(_pl_cal - pl_meas_valid)).numpy())
print(f'Calibrated scaling_factor = {float(scaling_factor_db.numpy()):+.4f} dB')
print(f'After  calibration : PL RMSE={_rmse_final:.2f} dB   MAE={_mae_final:.2f} dB')
print(f'PL RMSE improvement: {_rmse_init - _rmse_final:+.2f} dB')

# ── Save scalar offset for transfer to Sionna 2 DEM ──────────────────────────
import json as _json10, os as _os10
_SF_FILE = os.path.join(OUTPUT_DIR, f'scalar_offset_{int(FREQUENCY_HZ/1e6)}mhz.json')
_sf_save = {
    'meta': {
        'source'         : 'sionna019_differentiable_rt_fixed.ipynb Cell 10b',
        'frequency_mhz'  : float(FREQUENCY_HZ / 1e6),
        'scene'          : str(SCENE_XML),
        'tx_conducted_dbm': TX_CONDUCTED_DBM,
        'n_valid_pairs'  : int(rssi_sim_valid.shape[0]),
        'rmse_before_db' : float(_rmse_init),
        'rmse_after_db'  : float(_rmse_final),
    },
    'scaling_factor_db': float(scaling_factor_db.numpy()),
}
with open(_SF_FILE, 'w') as _f:
    _json10.dump(_sf_save, _f, indent=2)
print(f'\nScalar offset saved → {_SF_FILE}')
print(f'  scaling_factor_db = {float(scaling_factor_db.numpy()):+.4f} dB')
print(f'  Apply in Sionna 2 DEM: PL_sim_calibrated = PL_sim + {float(scaling_factor_db.numpy()):+.4f} dB')


---
## CELL 11b · Material Parameter Calibration — NVLabs Differentiable RT

**Purpose:** Reduce RMSE below the scalar-offset baseline by optimising physical material properties.

**Method (Hoydis et al. 2023, §IV):**
1. Pre-trace geometry once with `trace_paths()` — geometry is fixed, only EM changes
2. Inside `tf.GradientTape`: update materials → `compute_fields()` → RSSI → SMAPE loss
3. Adam applies gradients to ε_r, log(σ), S for each ITU material

**Key design decisions:**

| Decision | Reason |
|----------|--------|
| `trace_paths()` outside tape | Geometry is non-differentiable — only EM fields depend on materials |
| `compute_fields()` inside tape | This is differentiable w.r.t. material params in Sionna 0.19 |
| log(σ) parameterisation | σ spans 9 orders of magnitude (10⁻⁶ to 10⁷ S/m) — log-space is numerically stable |
| Receivers restored per batch | `compute_fields()` requires same receivers as `trace_paths()` — critical fix |
| Physical bounds | ε_r ∈ [1,100], σ ∈ [10⁻⁶,10⁷], S ∈ [0,1] enforced via `tf.clip_by_value` |

**Variables optimised:** ε_r, σ, S for each of: itu_concrete, itu_brick, itu_glass, itu_wood, itu_wet_ground, itu_water, itu_vegetation

| Parameter | Value |
|-----------|-------|
| Optimiser | Adam, LR = 0.01 |
| Steps | 200 |
| Loss | SMAPE on linear power |
| Pre-trace | `trace_paths()`, 50 RX/batch, 500k rays/batch |
| Output | Calibrated ε_r, σ, S table + RMSE before/after |

**After this cell:** Copy calibrated values into `sionna2_915mhz_dem_simulation.ipynb` Cell 4A  
to re-run the full DEM simulation with optimised materials.


In [ ]:
# ====================================================================
# CELL 11b — Material Parameter Calibration (NVLabs differentiable RT)
# Sionna 0.19 API: trace_paths() returns a tuple of 8 path objects;
# compute_fields(*traced_tuple) unpacks them correctly.
# ====================================================================
import time
import numpy as np
import tensorflow as tf

MAT_STEPS        = 300          # training iterations — converges by step ~50, 300 is safe margin
MAT_LR           = 5e-2         # initial LR (cosine-decayed to 1e-3)
MAT_BATCH        = 10           # receivers per batch — reduced from 100 to prevent GPU OOM
MAT_SAMPLES      = 2_000_000    # rays per trace — 2M ensures glass/vegetation edges hit consistently (Hoydis 2023b §4.2)
MAT_DEPTH        = CALIB_DEPTH
USE_TIKHONOV     = False         # knob: L2 reg toward ITU defaults
TIKHONOV_LAMBDA  = 0.01          # weight when USE_TIKHONOV=True
PRUNE_ZERO_GRADS = True          # remove zero-gradient vars after step 0
GRAD_CLIP_NORM   = 1.0           # per-variable gradient clip norm (Pascanu 2013) — prevents log_sig explosion
GRAD_ACCUM_SIZE  = 10            # batches per gradient micro-step — reduces peak GPU memory (114 batches / 10 = 12 micro-steps)

print('=' * 70)
print('CELL 11b — Material Parameter Calibration')
print('=' * 70)

# ── 1. Trainable variables per material ─────────────────────────────────
def _auto_bounds(key):
    """Derive calibration bounds from ITU defaults — works for any _ITU_P2040 key.
    eps: [default*0.3, default*4]  clamped to [1, 100]
    sig: [default*0.01, default*100] clamped to [1e-6, 1e7]
    S  : [0, min(0.95, default_S*3)]
    No manual maintenance needed — adding a key to _ITU_P2040 is enough."""
    if key is None or key not in _ITU_P2040:
        return None
    eps_d = _itu_at_freq(key, _freq_ghz)[0]
    sig_d = max(_itu_at_freq(key, _freq_ghz)[1], 1e-6)
    s_d   = _ITU_P2040[key][4]   # default scatter coefficient
    return (
        max(1.0,   eps_d * 0.3),  min(100.0, eps_d * 4.0),   # eps_r bounds
        max(1e-6,  sig_d * 0.01), min(1e7,   sig_d * 100.0), # sigma bounds
        0.0,                       min(0.95,  max(s_d * 3.0, 0.3)),  # S bounds
    )

mat11_vars = {}
mat11_init = {}
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    if _auto_bounds(key) is None:
        continue
    try:    eps0 = float(np.real(mat.relative_permittivity.numpy()))
    except: eps0 = _itu_at_freq(key, _freq_ghz)[0] if key else _DEFAULT_MAT[0]
    try:    sig0 = float(np.real(mat.conductivity.numpy()))
    except: sig0 = _itu_at_freq(key, _freq_ghz)[1] if key else _DEFAULT_MAT[2]
    sig0 = max(sig0, 1e-6)
    s0 = 0.0
    for _a in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, _a):
            try: s0 = float(getattr(mat, _a).numpy()); break
            except: pass
    mat11_init[mat_name] = {'eps': eps0, 'sig': sig0, 's': s0}
    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    mat11_vars[mat_name] = {
        'log_eps': tf.Variable(float(np.log(max(eps0 - 1.0, 1e-3))), dtype=tf.float32, name=f'{sn}_leps'),
        'log_sig': tf.Variable(float(np.log(sig0)),              dtype=tf.float32, name=f'{sn}_lsig'),
        's'      : tf.Variable(s0,                               dtype=tf.float32, name=f'{sn}_s'),
    }
print(f'Materials to calibrate : {len(mat11_vars)}')
for mn in mat11_vars:
    iv = mat11_init[mn]
    print(f'  {mn:<30}  eps_r={iv["eps"]:.3f}  sigma={iv["sig"]:.5f}  S={iv["s"]:.3f}')

# ── 2. Pre-trace geometry ─────────────────────────────────────────────────
# trace_paths() returns a tuple of 8 path objects:
#   (spec_paths, diff_paths, scat_paths, ris_paths,
#    spec_paths_tmp, diff_paths_tmp, scat_paths_tmp, ris_paths_tmp)
# Store the full tuple — unpack with * when calling compute_fields()
print(f'\nPre-tracing ({len(calib_receivers)} rx, batch={MAT_BATCH}, samples={MAT_SAMPLES:,}) ...')
# Detect trace_paths() parameter names — Sionna 0.19 vs 2.0
import inspect as _insp2
_tp_params = set(_insp2.signature(scene.trace_paths).parameters.keys())
_tr_cfg = {}
if 'max_depth'   in _tp_params: _tr_cfg['max_depth']   = MAT_DEPTH
if 'num_samples' in _tp_params: _tr_cfg['num_samples'] = MAT_SAMPLES
if 'los'         in _tp_params: _tr_cfg['los']         = True
if 'diffraction' in _tp_params: _tr_cfg['diffraction'] = True
# Sionna 0.19: reflection + scattering | Sionna 2.0: specular_reflection + diffuse_reflection
if 'reflection'          in _tp_params: _tr_cfg['reflection']          = True
if 'scattering'          in _tp_params: _tr_cfg['scattering']          = True
if 'specular_reflection' in _tp_params: _tr_cfg['specular_reflection'] = True
if 'diffuse_reflection'  in _tp_params: _tr_cfg['diffuse_reflection']  = True
print(f'trace_paths API params used: {list(_tr_cfg.keys())}')

_traced_list = []   # (paths_tuple, [rx_objects])
_meas_list   = []

for _b0 in range(0, len(calib_receivers), MAT_BATCH):
    _brx = calib_receivers[_b0 : _b0 + MAT_BATCH]
    _bm  = calib_rssi_meas[_b0 : _b0 + MAT_BATCH]
    for nm in list(scene.receivers.keys()):
        scene.remove(nm)
    for rx in _brx:
        scene.add(rx)
    try:
        _tp = scene.trace_paths(**_tr_cfg)   # returns 8-tuple
        _traced_list.append((_tp, list(_brx)))
        _meas_list.append(_bm)
    except Exception as e:
        print(f'  batch {_b0//MAT_BATCH+1}: trace_paths failed ({e})')
        continue
    if (_b0 // MAT_BATCH) % 5 == 0:
        print(f'  batch {_b0//MAT_BATCH+1}/{int(np.ceil(len(calib_receivers)/MAT_BATCH))} done')
print(f'Traced {len(_traced_list)} batches OK')

# ── 3. Apply variable values to scene materials ───────────────────────────
def _apply11():
    for mn, vd in mat11_vars.items():
        mat = scene.radio_materials.get(mn)
        if mat is None: continue
        eps_v = tf.exp(tf.clip_by_value(vd['log_eps'], tf.math.log(1e-3), tf.math.log(200.))) + 1.0
        sig_v = tf.exp(tf.clip_by_value(vd['log_sig'],
                       float(np.log(1e-6)), float(np.log(1e7))))
        s_v   = tf.clip_by_value(vd['s'],        0.0, 1.0)
        try:
            mat.relative_permittivity = eps_v
            mat.conductivity          = sig_v
        except: pass
        for _a in ('scattering_coefficient', 'scattering_coeff'):
            if hasattr(mat, _a):
                try: setattr(mat, _a, s_v); break
                except: pass

# ── 4. Evaluate all batches ────────────────────────────────────────────────
def _eval_all():
    _rs_all, _rm_all = [], []
    for (_tp, _brx), _bm in zip(_traced_list, _meas_list):
        # Restore this batch's receivers before compute_fields()
        for nm in list(scene.receivers.keys()):
            scene.remove(nm)
        for rx in _brx:
            scene.add(rx)
        try:
            _flds = scene.compute_fields(*_tp)
            # paths_to_rssi assumes a.shape[0]=n_rx; verify and fix if transposed
            _a = _flds.a
            if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
            _a = tf.cast(_a, tf.complex64)
            _pwr = tf.abs(_a)**2
            # shape can be (n_rx,n_tx,...) or (n_tx,n_rx,...) depending on API
            # use the axis that matches batch size
            _n_batch = len(_brx)
            if _pwr.shape[0] != _n_batch and _pwr.shape[1] == _n_batch:
                _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
            _nr = tf.shape(_pwr)[0]
            _p  = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
            # compute_fields returns normalized path gain — add TX power explicitly
            _rssi = 10.0*tf.math.log(_p+1e-30)/tf.math.log(10.0)+30.0+RX_EXTRA_GAIN_DB
            _rssi = tf.reshape(_rssi, [-1])
        except Exception as e:
            print(f'  _eval_all batch failed: {e}')
            continue
        _n  = min(len(_rssi), len(_bm))
        _rs = _rssi[:_n]
        _rm = tf.cast(_bm[:_n], tf.float32)
        _vm = tf.math.is_finite(_rs) & (_rs > -150.0)
        if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
            continue
        _rs_all.append(tf.boolean_mask(_rs, _vm))
        _rm_all.append(tf.boolean_mask(_rm, _vm))
    if not _rs_all:
        return None, None
    return tf.concat(_rs_all, 0), tf.concat(_rm_all, 0)

# ── 5. Baseline RMSE ──────────────────────────────────────────────────────
_apply11()
_rs0, _rm0 = _eval_all()
if _rs0 is not None:
    _pl0_sim  = TX_CONDUCTED_DBM - _rs0
    _pl0_meas = TX_CONDUCTED_DBM - _rm0
    _rmse11_init = float(tf.sqrt(tf.reduce_mean((_pl0_sim - _pl0_meas)**2)).numpy())
    _mae11_init  = float(tf.reduce_mean(tf.abs(_pl0_sim - _pl0_meas)).numpy())
    print(f'\nBefore material calib : PL RMSE={_rmse11_init:.2f} dB  MAE={_mae11_init:.2f} dB  N={len(_rs0)}')
else:
    _rmse11_init = 999.0
    print('ERROR: no valid pairs — check scene/receivers')

# ── 6. Training loop ──────────────────────────────────────────────────────
_all11_vars = []
for vd in mat11_vars.values():
    _all11_vars += [vd['log_eps'], vd['log_sig'], vd['s']]

# ITU initial values stored for optional Tikhonov regularisation
_itu11_init = {}
for mn, vd in mat11_vars.items():
    _itu11_init[mn] = {
        'log_eps': tf.constant(vd['log_eps'].numpy(), dtype=tf.float32),
        'log_sig': tf.constant(vd['log_sig'].numpy(), dtype=tf.float32),
        's':       tf.constant(vd['s'].numpy(),       dtype=tf.float32),
    }

_mat11_opt  = tf.keras.optimizers.Adam(learning_rate=MAT_LR)
_active_vars = _all11_vars          # will be pruned after step 0 if enabled
_pruned      = False

print(f'\nTraining material params ({MAT_STEPS} steps, LR={MAT_LR:.3f}→1e-3, '
      f'Tikhonov={USE_TIKHONOV}, prune={PRUNE_ZERO_GRADS})')
print('-' * 70)
t0 = time.time()
mat11_hist = {'step': [], 'loss': [], 'rmse': [], 'lr': []}

def _fields_to_power(tp_tuple):
    """Compute per-RX incoherent power from pre-traced paths.
    No @tf.function — scene receivers are swapped dynamically before each call,
    so XLA caching would freeze the old receiver set and produce zero fields."""
    _flds = scene.compute_fields(*tp_tuple)
    _a = _flds.a
    if isinstance(_a, tuple):
        _a = tf.complex(_a[0], _a[1])
    _a   = tf.cast(_a, tf.complex64)
    _pwr = tf.abs(_a) ** 2
    return _pwr

def _run_step(vars_to_opt):
    """One gradient step with gradient accumulation over GRAD_ACCUM_SIZE micro-batches.
    Reduces peak GPU memory vs single large tape over all 114 batches.
    Returns (grads, step_loss, rs_list, rm_list, n_ok)."""
    _step_rs, _step_rm = [], []
    _total_loss = tf.constant(0.0)
    _n_ok       = 0
    # Accumulated gradients — zero-initialised, same shape as vars
    _accum = [tf.zeros_like(v) for v in vars_to_opt]

    # Split traced batches into micro-groups of GRAD_ACCUM_SIZE
    _n_batches  = len(_traced_list)
    _group_size = max(1, GRAD_ACCUM_SIZE)

    for _g0 in range(0, _n_batches, _group_size):
        _group_tp   = _traced_list[_g0 : _g0 + _group_size]
        _group_meas = _meas_list  [_g0 : _g0 + _group_size]
        _micro_rs, _micro_rm = [], []
        _micro_loss = tf.constant(0.0)
        _micro_ok   = 0

        with tf.GradientTape() as tape:
            _apply11()
            for (_tp, _brx), _bm in zip(_group_tp, _group_meas):
                # Swap receivers for this batch (Python side-effect — outside XLA)
                for nm in list(scene.receivers.keys()):
                    scene.remove(nm)
                for rx in _brx:
                    scene.add(rx)
                try:
                    _pwr = _fields_to_power(_tp)   # XLA-compiled inner fn
                    _n_batch = len(_brx)
                    if _pwr.shape[0] != _n_batch and _pwr.shape[1] == _n_batch:
                        _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
                    _nr  = tf.shape(_pwr)[0]
                    _p   = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
                    _rssi = 10.0*tf.math.log(_p+1e-30)/tf.math.log(10.0)+30.0+RX_EXTRA_GAIN_DB
                    _rssi = tf.reshape(_rssi, [-1])
                    del _pwr, _p    # O3: free immediately — not needed after rssi
                except Exception:
                    continue
                _n  = min(len(_rssi), len(_bm))
                _rs = _rssi[:_n]
                _rm = tf.cast(_bm[:_n], tf.float32)
                _vm = tf.math.is_finite(_rs) & (_rs > -150.0)
                if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
                    continue
                _rs_v = tf.boolean_mask(_rs, _vm)
                _rm_v = tf.boolean_mask(_rm, _vm)
                _micro_loss = _micro_loss + smape_power_loss(_rs_v, _rm_v)
                _micro_rs.append(_rs_v)
                _micro_rm.append(_rm_v)
                _micro_ok += 1
                del _rs_v, _rm_v, _rssi  # O3: free per-batch tensors

            if _micro_ok > 0:
                _micro_loss = _micro_loss / float(_micro_ok)
                if USE_TIKHONOV:
                    _reg = tf.constant(0.0)
                    for mn, vd in mat11_vars.items():
                        _reg = _reg + tf.reduce_sum((vd['log_eps'] - _itu11_init[mn]['log_eps'])**2)
                        _reg = _reg + tf.reduce_sum((vd['log_sig'] - _itu11_init[mn]['log_sig'])**2)
                        _reg = _reg + tf.reduce_sum((vd['s']       - _itu11_init[mn]['s']      )**2)
                    _micro_loss = _micro_loss + TIKHONOV_LAMBDA * _reg

        # Accumulate gradients from this micro-group
        _grads_g = tape.gradient(_micro_loss, vars_to_opt)
        _accum = [
            _a + (tf.clip_by_norm(_g, GRAD_CLIP_NORM) if _g is not None else tf.zeros_like(_v))
            for _a, _g, _v in zip(_accum, _grads_g, vars_to_opt)
        ]
        del tape, _grads_g   # O3: release tape memory after each micro-group

        _total_loss = _total_loss + _micro_loss
        _step_rs.extend(_micro_rs)
        _step_rm.extend(_micro_rm)
        _n_ok += _micro_ok

    # Average accumulated gradients over actual valid batch count
    _n_ok_total = max(1, _n_ok)
    _accum = [_a / float(_n_ok_total) for _a in _accum]
    _total_loss = _total_loss / float(_n_ok_total)

    return _accum, _total_loss, _step_rs, _step_rm, _n_ok

for step in range(MAT_STEPS):
    # Cosine LR decay: LR_init → 1e-3
    import math
    _lr = 1e-3 + 0.5*(MAT_LR - 1e-3)*(1 + math.cos(math.pi * step / MAT_STEPS))
    _mat11_opt.learning_rate.assign(_lr)

    _grads, _step_loss, _step_rs, _step_rm, _n_ok = _run_step(_active_vars)

    if _n_ok == 0:
        print(f'  step {step:4d}: no valid batches — stopping'); break

    # ── After step 0: prune zero-gradient variables ───────────────
    if step == 0 and PRUNE_ZERO_GRADS and not _pruned:
        _zero_names, _active_names = [], []
        _new_active = []
        for v, g in zip(_active_vars, _grads):
            if g is None or float(tf.reduce_sum(tf.abs(g)).numpy()) < 1e-8:
                _zero_names.append(v.name)
            else:
                _new_active.append(v)
                _active_names.append(v.name)
        _active_vars = _new_active
        _mat11_opt   = tf.keras.optimizers.Adam(learning_rate=_lr)
        _pruned      = True
        print(f'  [prune] kept {len(_active_vars)}/{len(_all11_vars)} vars '
              f'with non-zero gradient')
        print(f'  [prune] active : {[n.split(":")[0] for n in _active_names]}')
        print(f'  [prune] removed: {[n.split(":")[0] for n in _zero_names]}')
        # Re-run step 0 with pruned vars so gradients match variables
        _grads, _step_loss, _step_rs, _step_rm, _n_ok = _run_step(_active_vars)
        if _n_ok == 0:
            print('  no valid batches after prune — stopping'); break
        lv = float(_step_loss.numpy()) * 100

    # Gradients already clipped inside _run_step (gradient accumulation loop)
    _mat11_opt.apply_gradients(zip(_grads, _active_vars))
    lv = float(_step_loss.numpy()) * 100
    # log every step — all columns same length (required for DataFrame)
    _pl_s   = TX_CONDUCTED_DBM - tf.concat(_step_rs, 0)
    _pl_m   = TX_CONDUCTED_DBM - tf.concat(_step_rm, 0)
    _rmse_v = float(tf.sqrt(tf.reduce_mean((_pl_s - _pl_m)**2)).numpy())
    mat11_hist['step'].append(step)
    mat11_hist['loss'].append(lv)
    mat11_hist['rmse'].append(_rmse_v)
    mat11_hist['lr'].append(_lr)
    for _mn, _vd in mat11_vars.items():
        _col_e = f'{_mn}_eps_r';  _col_s = f'{_mn}_sigma';  _col_sc = f'{_mn}_S'
        if _col_e  not in mat11_hist: mat11_hist[_col_e]  = []
        if _col_s  not in mat11_hist: mat11_hist[_col_s]  = []
        if _col_sc not in mat11_hist: mat11_hist[_col_sc] = []
        mat11_hist[_col_e].append(float(np.exp(np.clip(_vd['log_eps'].numpy(), np.log(1e-3), np.log(200.))) + 1.0))
        mat11_hist[_col_s].append(float(np.exp(_vd['log_sig'].numpy())))
        mat11_hist[_col_sc].append(float(_vd['s'].numpy()))

    if step % 50 == 0 or step == MAT_STEPS - 1:
        _nz = sum(1 for g in _grads if g is None or float(tf.reduce_sum(tf.abs(g))) < 1e-8)
        print(f'  step {step:4d}  PL RMSE={_rmse_v:5.2f} dB  SMAPE={lv:+7.2f}  '
              f'active={len(_active_vars)}/{len(_all11_vars)}  '
              f'zero_g={_nz}/{len(_active_vars)}  LR={_lr:.4f}  t={time.time()-t0:.0f}s')

print('-' * 70)
print(f'Done in {time.time()-t0:.1f}s')

# ── Save material calibration history ────────────────────────────────────────
import pandas as _pd11
_mat_hist_csv = os.path.join(OUTPUT_DIR, 'material_calib_history.csv')
_pd11.DataFrame(mat11_hist).to_csv(_mat_hist_csv, index=False)
print(f'Material history CSV saved → {_mat_hist_csv}')

# ── 7. Final RMSE ─────────────────────────────────────────────────────────
_apply11()
_rs_f, _rm_f = _eval_all()
if _rs_f is not None:
    _pl_f_sim  = TX_CONDUCTED_DBM - _rs_f
    _pl_f_meas = TX_CONDUCTED_DBM - _rm_f
    _rmse11_final = float(tf.sqrt(tf.reduce_mean((_pl_f_sim - _pl_f_meas)**2)).numpy())
    _mae11_final  = float(tf.reduce_mean(tf.abs(_pl_f_sim - _pl_f_meas)).numpy())
    print(f'\nAfter  material calib : PL RMSE={_rmse11_final:.2f} dB  MAE={_mae11_final:.2f} dB')
    print(f'PL RMSE improvement   : {_rmse11_init - _rmse11_final:+.2f} dB')

# ── 8. Calibrated parameter table ─────────────────────────────────────────
print(f'\n{"Material":<30} {"Param":<8} {"Init":>10} {"Final":>12} {"Delta":>8}')
print('-' * 72)
for mn, vd in mat11_vars.items():
    iv = mat11_init[mn]
    eps_c = float((tf.exp(tf.clip_by_value(vd['log_eps'], tf.math.log(1e-3), tf.math.log(200.))) + 1.0).numpy())
    sig_c = float(tf.exp(tf.clip_by_value(vd['log_sig'],
                  float(np.log(1e-6)), float(np.log(1e7)))).numpy())
    s_c   = float(tf.clip_by_value(vd['s'], 0.0, 1.0).numpy())
    print(f'  {mn:<28}  eps_r  {iv["eps"]:>10.3f}  {eps_c:>12.3f}  {eps_c-iv["eps"]:>+8.3f}')
    print(f'  {"":28}  sigma  {iv["sig"]:>10.5f}  {sig_c:>12.5f}  {sig_c-iv["sig"]:>+8.5f}')
    print(f'  {"":28}  S      {iv["s"]:>10.4f}  {s_c:>12.4f}  {s_c-iv["s"]:>+8.4f}')

# ── 9. Save calibrated parameters to JSON ─────────────────────────────
# File: calibrated_materials_915mhz.json
# Format matches Cell 4A MATERIAL_PROPS in sionna2_915mhz_dem_simulation.ipynb
# Strip "itu_" prefix and "_train" suffix → canonical material names.
# Load in Sionna 2 DEM Cell 4A for automatic transfer.
import json as _json_mod, os as _os_mod

_CALIB_FILE = os.path.join(OUTPUT_DIR, 'calibrated_materials_915mhz.json')

_calib_out = {
    'meta': {
        'source'       : 'sionna019_differentiable_rt_fixed.ipynb Cell 11b',
        'frequency_mhz': 915.0,
        'scene'        : str(SCENE_XML),
        'steps'        : MAT_STEPS,
        'rmse_before_db': float(_rmse11_init) if '_rmse11_init' in dir() else None,
        'rmse_after_db' : float(_rmse11_final) if '_rmse11_final' in dir() else None,
    },
    'materials': {}
}

for mn, vd in mat11_vars.items():
    # Canonical name: strip "itu_" prefix and "_train" suffix
    _canon = mn.replace('itu_', '').replace('_train', '')
    if _canon in _calib_out['materials']:
        continue   # keep first occurrence (non-_train)
    _eps_c = float((tf.exp(tf.clip_by_value(vd['log_eps'], tf.math.log(1e-3), tf.math.log(200.))) + 1.0).numpy())
    _sig_c = float(tf.exp(tf.clip_by_value(
        vd['log_sig'], float(np.log(1e-6)), float(np.log(1e7)))).numpy())
    _s_c   = float(tf.clip_by_value(vd['s'], 0.0, 1.0).numpy())
    _calib_out['materials'][_canon] = {
        'er'      : round(_eps_c, 4),
        'sigma'   : round(_sig_c, 6),
        'scatter' : round(_s_c,   4),
    }

with open(_CALIB_FILE, 'w') as _f:
    _json_mod.dump(_calib_out, _f, indent=2)

print(f'\nCalibrated materials saved → {_CALIB_FILE}')
print(f'  {len(_calib_out["materials"])} materials written:')
print(f'  {"Material":<20} {"er":>8} {"sigma":>12} {"scatter":>10}')
print('  ' + '-'*54)
for _mn, _mp in _calib_out['materials'].items():
    print(f'  {_mn:<20} {_mp["er"]:>8.3f} {_mp["sigma"]:>12.6f} {_mp["scatter"]:>10.4f}')


In [ ]:
n_mats   = len(trainable_mats)
colors   = plt.cm.tab10(np.linspace(0, 1, max(n_mats, 1)))
mat_list = list(trainable_mats.keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['step'], history['loss_db'], 'k-', lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss (dB)')
axes[0].set_title('Calibration Loss'); axes[0].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[1].plot(history['step'], history['eps_r'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[1].set_xlabel('Step'); axes[1].set_ylabel('ε_r')
axes[1].set_title('Relative Permittivity'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[2].plot(history['step'], history['log_sigma'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[2].set_xlabel('Step'); axes[2].set_ylabel('log σ (S/m)')
axes[2].set_title('Conductivity (log scale)'); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration_convergence.png'), dpi=150)
plt.show()

In [ ]:
print(f'{"Material":<30} {"Param":<8} {"Initial":>10} {"Calibrated":>12} {"Delta":>8}')
print('=' * 75)

calib_results = {}
for mn, mat in trainable_mats.items():
    orig = orig_params.get(mn, {})
    row  = {}
    for pname, attr_candidates, key in [
        ('eps_r', ['relative_permittivity'], 'eps_r'),
        ('sigma', ['conductivity'],           'sigma'),
        ('S',     ['scattering_coefficient', 'scattering_coeff'], 'S'),
    ]:
        init_val = orig.get(key, float('nan'))
        cal_val  = float('nan')
        for a_ in attr_candidates:
            if hasattr(mat, a_):
                try: cal_val = _safe(getattr(mat, a_)); break
                except: pass
        delta = cal_val - init_val if not np.isnan(init_val) else float('nan')
        print(f'  {mn.replace(_train_suffix,""):<28} {pname:<8} '
              f'{init_val:>10.4f} {cal_val:>12.4f} {delta:>+8.4f}')
        row[pname] = {'initial': init_val, 'calibrated': cal_val}
    calib_results[mn] = row

out_json = os.path.join(OUTPUT_DIR, 'calibration_results.json')
with open(out_json, 'w') as f:
    json.dump(calib_results, f, indent=2)
print(f'\nCalibration JSON saved to {out_json}')

---
## CELL 15 · Physics-Informed Residual MLP — Full RT Feature Set

Extracts **45 physics features** from Sionna 0.19 `compute_paths()` across 8 groups:
Power · Delay · Angular RX/TX · Path Types (LOS/specular/Lambertian/diffraction) · Vertex geometry · Scene geometry · Reference path loss.

Trains a **Deep Residual MLP** (Thrane et al. 2020 architecture) to predict the residual error `RSSI_meas − RSSI_sim`. Final output: `RSSI_final = RSSI_sim + Δ_RSSI`.

**References:** 3GPP TR 38.901 §7.3 · Thrane et al. 2020 IEEE TVT · Hoydis 2023 arXiv:2311.18558 · ITU-R P.1411-10 · Goldsmith 2005 §3

In [ ]:
# ====================================================================
# CELL 15 — Physics-Informed Residual MLP
# ====================================================================
# Extracts 45 physics features from pre-traced Sionna 0.19 paths and
# trains a deep residual MLP to predict RSSI_meas - RSSI_sim.
#
# Feature groups (45 total):
#   G1: Power        (5)  strongest path, total power, power ratio, dB spread, path count
#   G2: Delay        (5)  min/max/mean delay, RMS delay spread, coherence BW
#   G3: Angular RX   (5)  mean/std azimuth+elevation, angular spread
#   G4: Angular TX   (5)  same at TX side
#   G5: Path types   (5)  LOS flag, specular/diffuse/diffraction counts, LOS power fraction
#   G6: Vertex geom  (5)  mean/max interaction height, mean path length, reflections/diffractions
#   G7: Reference PL (5)  FSPL, distance, log-distance PL, TX-RX height diff, terrain correction
#   G8: Scene feats  (5)  building density, mean height, veg fraction, terrain std, urban flag
#   G9: Morphology   (5)  reserved zeros (nDSM not loaded in this notebook)
#
# Architecture: Deep Residual MLP (Thrane et al. 2020 IEEE TVT)
#   Input(45) → FC(128,ReLU) → FC(128,ReLU) + skip → FC(64,ReLU) → FC(1,Linear)
#
# Output: delta_dBm = RSSI_meas - RSSI_sim  (residual correction)
# Final:  RSSI_final = RSSI_sim + delta_pred
# ====================================================================
import numpy as np
import tensorflow as tf
import os, time, json as _json
from pyproj import Transformer as _TrR

print('=' * 70)
print('CELL 15 — Physics-Informed Residual MLP')
print(f'  Frequency : {FREQUENCY_HZ/1e6:.2f} MHz')
print('=' * 70)

_C15_STEPS     = 500
_C15_LR        = 1e-3
_C15_PATIENCE  = 80
_C15_SAVE      = os.path.join(OUTPUT_DIR, f'residual_mlp_{int(FREQUENCY_HZ/1e6)}mhz.npz')

# ── 1. Feature extraction from pre-traced paths ───────────────────────────────
def _extract_features(paths_tuple, rx_obj, tx_obj, scene_feats_8):
    """Extract 45 physics features for one receiver from a paths tuple."""
    feat = np.zeros(45, dtype=np.float32)
    try:
        # Unpack Sionna 0.19 trace_paths() tuple
        _sp = paths_tuple[0]   # specular paths object
        _dp = paths_tuple[2]   # diffuse/scatter paths
        _xp = paths_tuple[1]   # diffraction paths

        # ── G1: Power features (indices 0-4) ─────────────────────────────────
        try:
            _a = _sp.a
            if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
            _a = tf.cast(_a, tf.complex64)
            _pwr_paths = tf.abs(_a)**2
            _pwr_flat  = tf.reshape(_pwr_paths, [-1]).numpy()
            _pwr_flat  = _pwr_flat[_pwr_flat > 1e-30]
            if len(_pwr_flat) > 0:
                feat[0] = float(np.max(_pwr_flat))                          # strongest path power
                feat[1] = float(np.sum(_pwr_flat))                          # total received power
                feat[2] = float(feat[0] / (feat[1] + 1e-30))               # dominant path ratio
                feat[3] = float(10*np.log10(np.max(_pwr_flat)+1e-30) -
                                10*np.log10(np.min(_pwr_flat)+1e-30))       # power spread dB
                feat[4] = float(min(len(_pwr_flat), 500)) / 500.0           # path count norm
        except Exception: pass

        # ── G2: Delay features (indices 5-9) ─────────────────────────────────
        try:
            _tau = _sp.tau.numpy().flatten()
            _tau = _tau[_tau > 0]
            if len(_tau) > 0:
                _pwr_d = _pwr_flat[:len(_tau)] if len(_pwr_flat) >= len(_tau) else _pwr_flat
                _w = _pwr_d / (_pwr_d.sum() + 1e-30)
                feat[5] = float(np.min(_tau)) * 1e6                        # min delay µs
                feat[6] = float(np.max(_tau)) * 1e6                        # max delay µs
                feat[7] = float(np.average(_tau[:len(_w)], weights=_w)) * 1e6  # mean delay µs
                _rms_ds = float(np.sqrt(np.average(
                    (_tau[:len(_w)] - feat[7]*1e-6)**2, weights=_w))) * 1e9 # RMS delay ns
                feat[8] = _rms_ds / 1000.0                                  # RMS delay norm
                feat[9] = float(1.0 / (2 * max(_rms_ds*1e-9, 1e-9)) / 1e6) / 100.0  # coherence BW
        except Exception: pass

        # ── G3: Angular RX features (indices 10-14) ──────────────────────────
        try:
            _phi_r   = _sp.phi_r.numpy().flatten()
            _theta_r = _sp.theta_r.numpy().flatten()
            _phi_r   = _phi_r[np.isfinite(_phi_r)]
            _theta_r = _theta_r[np.isfinite(_theta_r)]
            if len(_phi_r) > 0:
                feat[10] = float(np.mean(_phi_r))   / np.pi               # mean az RX norm
                feat[11] = float(np.std(_phi_r))    / np.pi               # std az RX norm
                feat[12] = float(np.mean(_theta_r)) / np.pi               # mean el RX norm
                feat[13] = float(np.std(_theta_r))  / np.pi               # std el RX norm
                feat[14] = float(np.sqrt(np.std(_phi_r)**2 +
                                          np.std(_theta_r)**2)) / np.pi   # angular spread RX
        except Exception: pass

        # ── G4: Angular TX features (indices 15-19) ──────────────────────────
        try:
            _phi_t   = _sp.phi_t.numpy().flatten()
            _theta_t = _sp.theta_t.numpy().flatten()
            _phi_t   = _phi_t[np.isfinite(_phi_t)]
            _theta_t = _theta_t[np.isfinite(_theta_t)]
            if len(_phi_t) > 0:
                feat[15] = float(np.mean(_phi_t))   / np.pi
                feat[16] = float(np.std(_phi_t))    / np.pi
                feat[17] = float(np.mean(_theta_t)) / np.pi
                feat[18] = float(np.std(_theta_t))  / np.pi
                feat[19] = float(np.sqrt(np.std(_phi_t)**2 +
                                          np.std(_theta_t)**2)) / np.pi
        except Exception: pass

        # ── G5: Path type features (indices 20-24) ───────────────────────────
        try:
            _n_spec  = int(_pwr_flat.shape[0]) if len(_pwr_flat) > 0 else 0
            _n_diff  = 0
            _n_scat  = 0
            _has_los = 0.0
            try:
                _los_pwr = float(tf.reduce_sum(tf.abs(tf.cast(_sp.a, tf.complex64))**2).numpy())
                _has_los = 1.0 if _los_pwr > feat[1] * 0.5 else 0.0
            except: pass
            try:
                _da = _dp.a
                if isinstance(_da, tuple): _da = tf.complex(_da[0], _da[1])
                _n_scat = int(tf.size(_da).numpy())
            except: pass
            try:
                _xa = _xp.a
                if isinstance(_xa, tuple): _xa = tf.complex(_xa[0], _xa[1])
                _n_diff = int(tf.size(_xa).numpy())
            except: pass
            feat[20] = _has_los
            feat[21] = float(min(_n_spec, 1000)) / 1000.0
            feat[22] = float(min(_n_scat, 1000)) / 1000.0
            feat[23] = float(min(_n_diff, 500))  / 500.0
            feat[24] = float(_has_los) * feat[2]   # LOS power fraction
        except Exception: pass

        # ── G6: Vertex geometry features (indices 25-29) ─────────────────────
        try:
            _vtx = _sp.vertices.numpy()   # shape: [n_paths, max_depth, 3]
            _vtx_flat = _vtx.reshape(-1, 3)
            _valid = _vtx_flat[np.any(_vtx_flat != 0, axis=1)]
            if len(_valid) > 0:
                feat[25] = float(np.mean(np.abs(_valid[:, 2]))) / 50.0    # mean interaction height
                feat[26] = float(np.max(np.abs(_valid[:, 2])))  / 100.0   # max interaction height
                _path_lens = np.sqrt(np.sum(np.diff(_vtx[:,:,:2], axis=1)**2, axis=2))
                feat[27] = float(np.mean(_path_lens[_path_lens > 0])) / 1000.0  # mean path length km
                feat[28] = float(np.sum(_path_lens > 0.1)) / float(max(_vtx.shape[0], 1)) / 10.0
                feat[29] = float(np.max(_path_lens)) / 2000.0             # max path length norm
        except Exception: pass

        # ── G7: Reference path loss features (indices 30-34) ─────────────────
        try:
            _rx_pos = np.array([float(rx_obj.position[i].numpy()
                                if hasattr(rx_obj.position[i], 'numpy')
                                else rx_obj.position[i]) for i in range(3)])
            _tx_pos = np.array([float(tx_obj.position[i].numpy()
                                if hasattr(tx_obj.position[i], 'numpy')
                                else tx_obj.position[i]) for i in range(3)])
            _dist3d = float(np.linalg.norm(_rx_pos - _tx_pos))
            _dist2d = float(np.linalg.norm(_rx_pos[:2] - _tx_pos[:2]))
            _lam    = 3e8 / FREQUENCY_HZ
            _fspl   = 20*np.log10(max(_dist3d,1)) + 20*np.log10(FREQUENCY_HZ) - 147.55
            feat[30] = _fspl / 150.0                                       # FSPL norm
            feat[31] = _dist2d / 2000.0                                    # 2D dist norm
            feat[32] = float(10 * np.log10(max(_dist2d, 1))) / 40.0       # log-distance
            feat[33] = float(_rx_pos[2] - _tx_pos[2]) / 100.0             # height diff norm
            feat[34] = float(abs(_rx_pos[2])) / 50.0                      # RX height AGL norm
        except Exception: pass

        # ── G8: Scene features (indices 35-39) — from compute_scene_features ─
        feat[35:40] = scene_feats_8[:5]

        # ── G9: Morphology (indices 40-44) — reserved zeros ──────────────────
        feat[40:45] = 0.0

    except Exception as _ex:
        pass   # return zeros on any failure — robust to API changes

    return feat

# ── 2. Build feature matrix from _traced_list ────────────────────────────────
if '_traced_list' not in dir() or len(_traced_list) == 0:
    print('[CELL 15] _traced_list not found — run Cell 11b first.')
    raise RuntimeError('Run Cell 11b first to generate _traced_list')

_tx_obj = list(scene.transmitters.values())[0]

# Scene features (8-D) — reuse from Cell 16 if available
if '_SCENE_FEATS_TF' in dir():
    _sf8 = _SCENE_FEATS_TF.numpy().flatten()[:8]
else:
    _sf8 = np.zeros(8, dtype=np.float32)

print(f'Extracting features from {len(_traced_list)} batches ...')
_t0_feat = time.time()
_X15, _y15, _rx_names = [], [], []

for _tp_batch, _brx in _traced_list:
    _bm_idx = [i for i, r in enumerate(calib_receivers) if r in _brx]
    _bm = calib_rssi_meas[_bm_idx].numpy() if hasattr(calib_rssi_meas, 'numpy')           else np.array(calib_rssi_meas)[_bm_idx]

    for _ri, (_rx, _meas) in enumerate(zip(_brx, _bm)):
        # Slice paths for this receiver index
        try:
            _feat = _extract_features(_tp_batch, _rx, _tx_obj, _sf8)
        except Exception:
            _feat = np.zeros(45, dtype=np.float32)

        # Simulated RSSI for this receiver from scalar-offset baseline
        _sim_v = float('nan')
        if 'rssi_sim_cached' in dir():
            _rx_idx = next((i for i, r in enumerate(calib_receivers) if r.name == _rx.name), None)
            if _rx_idx is not None and _rx_idx < len(rssi_sim_cached):
                _sim_v = float(rssi_sim_cached[_rx_idx])
        if np.isnan(_sim_v): continue

        _residual = float(_meas) - _sim_v
        if not np.isfinite(_residual): continue
        if abs(_residual) > 40: continue   # outlier rejection: >40 dB residual

        _X15.append(_feat)
        _y15.append(_residual)
        _rx_names.append(_rx.name)

_X15 = np.array(_X15, dtype=np.float32)
_y15 = np.array(_y15, dtype=np.float32)
print(f'  Features extracted: {len(_X15)} receivers  ({time.time()-_t0_feat:.1f}s)')
print(f'  Residual: mean={_y15.mean():.2f} dB  std={_y15.std():.2f} dB  '
      f'min={_y15.min():.1f}  max={_y15.max():.1f}')

if len(_X15) < 10:
    print('[WARN] Too few samples for MLP — check _traced_list and rssi_sim_cached')

# ── 3. Normalise features ─────────────────────────────────────────────────────
_X15_mean = _X15.mean(axis=0)
_X15_std  = np.where(_X15.std(axis=0) > 1e-8, _X15.std(axis=0), 1.0)
_X15_norm = (_X15 - _X15_mean) / _X15_std

# ── 4. Train/test split ───────────────────────────────────────────────────────
np.random.seed(42)
_idx  = np.random.permutation(len(_X15_norm))
_n_tr = int(0.8 * len(_idx))
_tr, _te = _idx[:_n_tr], _idx[_n_tr:]
_Xtr, _Xte = _X15_norm[_tr], _X15_norm[_te]
_ytr, _yte = _y15[_tr],      _y15[_te]
print(f'  Train: {len(_tr)}  Test: {len(_te)}')

# ── 5. Deep Residual MLP (Thrane et al. 2020) ────────────────────────────────
def _build_residual_mlp(n_feat=45):
    _inp  = tf.keras.Input(shape=(n_feat,), name='physics_features')
    _h1   = tf.keras.layers.Dense(128, activation='relu',
                kernel_regularizer=tf.keras.regularizers.l2(1e-4))(_inp)
    _h1   = tf.keras.layers.Dropout(0.2)(_h1)
    _h2   = tf.keras.layers.Dense(128, activation='relu',
                kernel_regularizer=tf.keras.regularizers.l2(1e-4))(_h1)
    _h2   = tf.keras.layers.Dropout(0.2)(_h2)
    # Residual skip connection: h1 + h2 (both 128-dim)
    _skip = tf.keras.layers.Add()([_h1, _h2])
    _h3   = tf.keras.layers.Dense(64, activation='relu')(_skip)
    _out  = tf.keras.layers.Dense(1, activation='linear', name='residual_db')(_h3)
    return tf.keras.Model(_inp, _out, name='ResidualMLP_C15')

_mlp15 = _build_residual_mlp(45)
_mlp15.compile(
    optimizer=tf.keras.optimizers.Adam(_C15_LR),
    loss='mse',
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)
print(f'  Model params: {_mlp15.count_params():,}')

# ── 6. Training ───────────────────────────────────────────────────────────────
_t0_train = time.time()
print(f'Training residual MLP ({_C15_STEPS} epochs) ...')
_hist15 = _mlp15.fit(
    _Xtr, _ytr,
    validation_data=(_Xte, _yte),
    epochs=_C15_STEPS,
    batch_size=max(8, len(_Xtr)//8),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_rmse', patience=_C15_PATIENCE,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=30,
            min_lr=1e-5, verbose=0),
    ],
    verbose=0
)
_best15 = int(np.argmin(_hist15.history['val_rmse'])) + 1
print(f'  Best epoch : {_best15}  |  '
      f'train RMSE={_hist15.history["rmse"][_best15-1]:.2f} dB  '
      f'val RMSE={_hist15.history["val_rmse"][_best15-1]:.2f} dB  '
      f'({time.time()-_t0_train:.1f}s)')

# Print every 50 epochs
for ep in range(0, len(_hist15.history['rmse']), 50):
    print(f'  epoch {ep+1:4d}  train={_hist15.history["rmse"][ep]:.2f} dB'
          f'  val={_hist15.history["val_rmse"][ep]:.2f} dB')

# ── 7. Evaluation ─────────────────────────────────────────────────────────────
_pred15   = _mlp15.predict(_Xte, verbose=0).flatten()
_rssi_sim_te = np.array([float(rssi_sim_cached[list(calib_receivers).index(
    next(r for r in calib_receivers if r.name == _rx_names[i]))])
    for i in _te], dtype=np.float32) if 'rssi_sim_cached' in dir() else np.zeros(len(_te))
_rssi_final15 = _rssi_sim_te + _pred15
_rssi_meas_te = _yte + _rssi_sim_te

_rmse_raw = float(np.sqrt(np.mean((_rssi_sim_te - _rssi_meas_te)**2)))
_rmse_mlp = float(np.sqrt(np.mean((_rssi_final15 - _rssi_meas_te)**2)))
_bias_mlp = float(np.mean(_rssi_final15 - _rssi_meas_te))
_mae_mlp  = float(np.mean(np.abs(_rssi_final15 - _rssi_meas_te)))

print('\n' + '='*60)
print('CELL 15 — RESULTS')
print('='*60)
print(f'  RT only RMSE           : {_rmse_raw:>7.2f} dB')
print(f'  RT + Residual MLP RMSE : {_rmse_mlp:>7.2f} dB  (Δ={_rmse_raw-_rmse_mlp:+.2f} dB)')
print(f'  Residual MLP bias      : {_bias_mlp:>7.2f} dB')
print(f'  Residual MLP MAE       : {_mae_mlp:>7.2f} dB')
print(f'  Improvement            : {(1-_rmse_mlp/_rmse_raw)*100:.1f}%')

# ── 8. Save weights + normalisation stats ────────────────────────────────────
_w15 = {f'w{i}': v.numpy() for i, v in enumerate(_mlp15.trainable_variables)}
_w15['feat_mean'] = _X15_mean
_w15['feat_std']  = _X15_std
_w15['frequency_hz'] = np.array([FREQUENCY_HZ])
np.savez(_C15_SAVE, **_w15)
print(f'  Weights saved → {_C15_SAVE}')
print('Done.')


---
## CELL 16 · MaterialMLP — Scene-Conditioned Material Calibration

**Purpose:** End-to-end differentiable MLP that maps `(material_one_hot + scene_features) → (ε_r, σ, S)`.

**Key advantage over Cell 11b:** The MLP *generalises* — once trained on this scene, call
`predict_materials(scene_feats)` on any new scene (Sionna 2 DEM, London, etc.) **without re-running RT**.

**Architecture:**
- Input: 14 features (6 material one-hot + 8 scene features)
- Layer 1: Dense(64) → BatchNorm → ReLU
- Layer 2: Dense(32) → BatchNorm → ReLU
- Output heads: Dense(1) each for ε_r, log(σ), S (physical bounds via sigmoid/exp)
- Total trainable params: ~7 360

**Run order:** Cell 2 → Cell 4 → Cell 6 → Cell 7 → Cell 8b → Cell 11b → **Cell 16**

**Saves:** `calibrated_materials_mlp_915mhz.json` + `material_mlp_weights.npz`

In [ ]:
# ====================================================================
# CELL 16 — MaterialMLP: Scene-Conditioned Material Calibration
# ====================================================================
# Architecture:
#   [material_one_hot (6) + scene_features (8)] → MLP → [ε_r, σ, S]
#
# Gradient flow (end-to-end differentiable):
#   Loss → RSSI_sim → compute_fields() → ε_r,σ,S → MLP weights
#
# Why MLP instead of tf.Variable:
#   - tf.Variable: calibrates for THIS scene only
#   - MLP: learns a mapping (scene_context → materials) that
#          GENERALISES — run inference on Sionna 2 DEM by computing
#          scene features and calling mlp.predict(), no RT needed
#
# Requires: _traced_list from Cell 11b (pre-traced paths, reused)
# Saves   : calibrated_materials_mlp_915mhz.json  (same format as
#           Cell 11b) + material_mlp_weights.npz (MLP weights)
#
# Transfer to Sionna 2 DEM:
#   1. Compute scene features for new scene (function below)
#   2. Load MLP weights
#   3. Call predict_materials(scene_feats) → ε_r, σ, S
#   4. Apply to Sionna 2 materials — no RT calibration needed
# ====================================================================

import tensorflow as tf
import numpy as np
import json, time, os
from tensorflow.keras.layers import Dense, BatchNormalization

# ── Configuration ─────────────────────────────────────────────────────
_C16_STEPS    = 300
_C16_LR       = 5e-3
_C16_PATIENCE = 80
_C16_SAVE_DIR = OUTPUT_DIR

# ── Materials to calibrate (dominant urban — proven active in Cell 11b)
_C16_MATS = ['concrete', 'brick', 'glass', 'wet_ground',
             'vegetation', 'water']
_N_MAT = len(_C16_MATS)

print('=' * 70)
print('CELL 16 — MaterialMLP: Scene-Conditioned Material Calibration')
print('=' * 70)
print(f'Materials : {_C16_MATS}')
print(f'Steps     : {_C16_STEPS}  LR={_C16_LR}  patience={_C16_PATIENCE}')

# ── Step 1: Compute scene features ────────────────────────────────────
# These 8 features describe the scene context for the MLP.
# They must be computable for ANY new scene (Sionna 2 DEM, London, etc.)

def compute_scene_features(freq_hz, ndsm_data=None, tx_pos=None,
                            rx_positions=None):
    """
    Compute 8 normalised scene-level features.
    All inputs are optional — falls back to safe defaults.
    """
    feat = np.zeros(8, dtype=np.float32)

    # 0: Frequency (GHz / 100) — normalised
    feat[0] = float(freq_hz) / 100e9

    # 1-3: nDSM statistics
    if ndsm_data is not None:
        _h = ndsm_data[ndsm_data > 0.5].astype(np.float32)
        feat[1] = float(np.mean(_h))   / 20.0    if len(_h) > 0 else 0.0
        feat[2] = float(np.std(_h))    / 10.0    if len(_h) > 0 else 0.0
        feat[3] = float(np.mean(ndsm_data > 2.0))          # building density
    else:
        print('[WARN] nDSM not available — scene features set to 0 (city-agnostic fallback)')
        feat[1], feat[2], feat[3] = 0.0, 0.0, 0.0   # zero fallback — no city assumption

    # 4: TX height (normalised)
    feat[4] = float(tx_pos[2]) / 100.0 if tx_pos is not None else 0.0

    # 5-6: Mean/std RX height
    if rx_positions is not None and len(rx_positions) > 0:
        _rh = np.array([float(r.position.numpy()[2])
                        for r in rx_positions], dtype=np.float32)
        feat[5] = float(np.mean(_rh)) / 10.0
        feat[6] = float(np.std(_rh))  / 5.0
    else:
        feat[5], feat[6] = 0.0, 0.0   # zero fallback — no city assumption

    # 7: Urban density proxy (N_rx / scene_area)
    feat[7] = min(len(rx_positions) / 1200.0, 1.0) \
              if rx_positions is not None else 1.0

    return feat

# Compute for current scene
_tx_obj   = list(scene.transmitters.values())[0]
_tx_pos_c = np.array(_tx_obj.position.numpy(), dtype=np.float32)
_scene_feats = compute_scene_features(
    freq_hz      = float(FREQUENCY_HZ),
    ndsm_data    = _ndsm_data if '_ndsm_data' in dir() else None,
    tx_pos       = _tx_pos_c,
    rx_positions = calib_receivers,
)
print(f'\nScene features (8):')
_sf_names = ['freq_norm','ndsm_mean','ndsm_std','bld_density',
             'tx_height','rx_h_mean','rx_h_std','urban_density']
for i, (n, v) in enumerate(zip(_sf_names, _scene_feats)):
    print(f'  {i}: {n:<16} {v:+.4f}')

_SCENE_FEATS_TF = tf.constant(_scene_feats[np.newaxis, :],
                               dtype=tf.float32)    # [1, 8]
_MAT_IDS_TF     = tf.eye(_N_MAT, dtype=tf.float32) # [n_mat, n_mat]

# ── Step 2: ITU default values (for bounds and comparison) ────────────
_ITU_DEFAULTS = {
    'concrete'  : (5.24,  0.130,  0.15),
    'brick'     : (3.91,  0.024,  0.10),
    'glass'     : (6.27,  0.012,  0.05),
    'wet_ground': (30.0,  0.150,  0.05),
    'vegetation': (1.30,  0.001,  0.75),
    'water'     : (81.0,  0.500,  0.02),
}

# ── Step 3: MaterialMLP definition ────────────────────────────────────
class MaterialMLP(tf.keras.Model):
    """
    Maps (material_one_hot + scene_features) → (ε_r, σ, S).
    Input dim : n_mat + n_scene_feats = 6 + 8 = 14
    Architecture: FC(64)→BN→ReLU→FC(32)→BN→ReLU→3 heads
    """
    def __init__(self, n_mat, n_scene=8):
        super().__init__()
        self.n_mat  = n_mat
        inp = n_mat + n_scene
        self.fc1    = Dense(64, activation=None,
                            kernel_regularizer=
                            tf.keras.regularizers.l2(1e-4))
        self.bn1    = BatchNormalization()
        self.fc2    = Dense(32, activation=None,
                            kernel_regularizer=
                            tf.keras.regularizers.l2(1e-4))
        self.bn2    = BatchNormalization()
        # Three separate output heads
        self.eps_head = Dense(1, name='eps_r')
        self.sig_head = Dense(1, name='log_sigma')
        self.s_head   = Dense(1, name='scatter')

    def call(self, mat_ids, scene_feats, training=False):
        """
        mat_ids    : [n_mat, n_mat]  one-hot
        scene_feats: [1, n_scene]    broadcast
        Returns    : (eps_r, sigma, s) each [n_mat]
        """
        # Broadcast scene features to each material
        sf = tf.tile(scene_feats, [self.n_mat, 1])     # [n_mat, n_scene]
        x  = tf.concat([mat_ids, sf], axis=1)          # [n_mat, n_mat+n_scene]

        x  = tf.nn.relu(self.bn1(self.fc1(x), training=training))
        x  = tf.nn.relu(self.bn2(self.fc2(x), training=training))

        # Physical bounds via smooth activations
        eps_r = 1.0  + 99.0  * tf.sigmoid(self.eps_head(x))   # [1,100]
        sigma = tf.exp(tf.clip_by_value(self.sig_head(x),
                       -14.0, 16.0))                            # [1e-6,1e7]
        s     = tf.sigmoid(self.s_head(x))                      # [0,1]

        return (tf.squeeze(eps_r, axis=1),   # [n_mat]
                tf.squeeze(sigma, axis=1),
                tf.squeeze(s,     axis=1))

    def predict_materials(self, scene_feats):
        """Inference only — returns numpy dict for a given scene."""
        sf = tf.constant(scene_feats[np.newaxis, :], dtype=tf.float32)
        e, sg, s = self(tf.eye(self.n_mat), sf, training=False)
        return {_C16_MATS[i]: {
                    'er':      float(e[i].numpy()),
                    'sigma':   float(sg[i].numpy()),
                    'scatter': float(s[i].numpy())}
                for i in range(self.n_mat)}

_mat_mlp   = MaterialMLP(_N_MAT)
_opt_c16   = tf.keras.optimizers.Adam(_C16_LR)
print(f'\nMaterialMLP built — trainable params: '
      f'{sum(np.prod(v.shape) for v in _mat_mlp.trainable_variables)}')

# ── Step 4: Apply MLP output to scene ─────────────────────────────────
def _apply_mlp_materials(eps_r, sigma, s):
    """Write MLP-predicted params into scene material objects."""
    for mi, base in enumerate(_C16_MATS):
        for suffix in ['', '_train']:
            full = f'itu_{base}{suffix}'
            if full in scene.objects:
                obj = scene.get(full)
                try:
                    obj.relative_permittivity  = eps_r[mi]
                    obj.conductivity           = sigma[mi]
                    obj.scattering_coefficient = s[mi]
                except Exception:
                    pass

# ── Step 5: Training loop ─────────────────────────────────────────────
best_loss  = 1e9
best_w     = None
patience   = 0
hist_c16   = {'step': [], 'loss': [], 'rmse': []}

print(f'\nTraining MaterialMLP ({_C16_STEPS} steps, LR={_C16_LR})')
print('-' * 65)
t0 = time.time()

for step in range(_C16_STEPS):
    _step_rs, _step_rm = [], []
    _step_loss = tf.constant(0.0)
    _n_ok = 0

    with tf.GradientTape() as tape:
        # 1. MLP predicts material params from scene context
        eps_r, sigma, s = _mat_mlp(_MAT_IDS_TF, _SCENE_FEATS_TF,
                                    training=True)

        # 2. Apply predicted params to scene (inside tape)
        _apply_mlp_materials(eps_r, sigma, s)

        # 3. compute_fields for each pre-traced batch
        for (_tp, _brx), _bm in zip(_traced_list, _meas_list):
            for nm in list(scene.receivers.keys()):
                scene.remove(nm)
            for rx in _brx:
                scene.add(rx)
            try:
                _flds = scene.compute_fields(*_tp)
                _a    = _flds.a
                if isinstance(_a, tuple):
                    _a = tf.complex(_a[0], _a[1])
                _a    = tf.cast(_a, tf.complex64)
                _pwr  = tf.abs(_a)**2
                _nb   = len(_brx)
                if _pwr.shape[0] != _nb and _pwr.shape[1] == _nb:
                    _pwr = tf.transpose(_pwr,
                               [1,0]+list(range(2,len(_pwr.shape))))
                _nr   = tf.shape(_pwr)[0]
                _p    = tf.cast(tf.reduce_sum(
                            tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
                _rssi = (10.0*tf.math.log(_p+1e-30)/tf.math.log(10.)
                         + 30.0 + RX_EXTRA_GAIN_DB)
                _rssi = tf.reshape(_rssi, [-1])
            except Exception:
                continue

            _n   = min(len(_rssi), len(_bm))
            _rs  = _rssi[:_n]
            _rm  = tf.cast(_bm[:_n], tf.float32)
            _vm  = tf.math.is_finite(_rs) & (_rs > -150.0)
            if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
                continue
            _rs_v = tf.boolean_mask(_rs, _vm)
            _rm_v = tf.boolean_mask(_rm, _vm)
            _step_loss = _step_loss + smape_power_loss(_rs_v, _rm_v)
            _step_rs.append(_rs_v)
            _step_rm.append(_rm_v)
            _n_ok += 1

        if _n_ok > 0:
            _step_loss = _step_loss / float(_n_ok)

    if _n_ok == 0:
        print(f'  step {step}: no valid batches — stopping'); break

    # 4. Gradient flows back through compute_fields → MLP weights
    _grads = tape.gradient(_step_loss, _mat_mlp.trainable_variables)
    _grads = [g if g is not None else tf.zeros_like(v)
              for g, v in zip(_grads, _mat_mlp.trainable_variables)]
    _opt_c16.apply_gradients(zip(_grads, _mat_mlp.trainable_variables))

    lv = float(_step_loss.numpy()) * 100
    hist_c16['step'].append(step)
    hist_c16['loss'].append(lv)

    # Early stopping
    if lv < best_loss:
        best_loss = lv
        best_w    = _mat_mlp.get_weights()
        patience  = 0
    else:
        patience += 1

    if step % 50 == 0 or step == _C16_STEPS - 1:
        if _step_rs:
            _rmse = float(tf.sqrt(tf.reduce_mean(
                (tf.concat(_step_rs,0)-tf.concat(_step_rm,0))**2)).numpy())
            hist_c16['rmse'].append(_rmse)
            print(f'  step {step:4d}  RMSE={_rmse:.2f} dB  '
                  f'SMAPE={lv:.2f}  patience={patience}  '
                  f't={time.time()-t0:.0f}s')

    if patience >= _C16_PATIENCE and step > 100:
        print(f'  Early stop at step {step}  best_loss={best_loss:.2f}')
        break

_mat_mlp.set_weights(best_w)
print(f'\nDone in {time.time()-t0:.1f}s')

# ── Step 6: Read calibrated values from MLP ───────────────────────────
_calib_c16 = _mat_mlp.predict_materials(_scene_feats)

print(f'\n{"Material":<16} {"ε_r ITU":>8} {"ε_r cal":>8} '
      f'{"σ ITU":>10} {"σ cal":>10} {"S ITU":>7} {"S cal":>7}')
print('-' * 72)
for mn in _C16_MATS:
    d    = _calib_c16[mn]
    itu  = _ITU_DEFAULTS[mn]
    print(f'  {mn:<14} {itu[0]:>8.3f} {d["er"]:>8.3f} '
          f'{itu[1]:>10.5f} {d["sigma"]:>10.5f} '
          f'{itu[2]:>7.3f} {d["scatter"]:>7.3f}')

# ── Step 7: Save JSON + MLP weights ───────────────────────────────────
_out_json = {
    'meta': {
        'source'       : 'sionna019_differentiable_rt_fixed.ipynb Cell 16',
        'model'        : 'MaterialMLP (scene-conditioned)',
        'frequency_mhz': float(FREQUENCY_HZ / 1e6),
        'scene'        : str(SCENE_XML),
        'steps'        : _C16_STEPS,
        'scene_features': _scene_feats.tolist(),
        'scene_feat_names': _sf_names,
    },
    'materials': _calib_c16
}
_json_path = os.path.join(_C16_SAVE_DIR,
                           f'calibrated_materials_mlp_{int(FREQUENCY_HZ/1e6)}mhz.json')
with open(_json_path, 'w') as _f:
    json.dump(_out_json, _f, indent=2)

# Save MLP weights for inference on new scenes
_npz_path = os.path.join(_C16_SAVE_DIR, 'material_mlp_weights.npz')
np.savez(_npz_path,
         **{f'w{i}': w for i, w in
            enumerate(_mat_mlp.get_weights())},
         scene_feats=_scene_feats)
print(f'\nJSON saved  → {_json_path}')
print(f'MLP weights → {_npz_path}')

# ── Step 8: Inference example for Sionna 2 DEM ────────────────────────
print('\n' + '=' * 65)
print('INFERENCE ON NEW SCENE (Sionna 2 DEM example)')
print('=' * 65)
print('''
# In sionna2_915mhz_dem_simulation.ipynb, Cell 4A:
#
# 1. Load MLP weights
import numpy as np, tensorflow as tf
from tensorflow.keras.layers import Dense, BatchNormalization

_w   = np.load('material_mlp_weights.npz', allow_pickle=True)
_mlp = MaterialMLP(n_mat=6)       # same architecture
_mlp(_MAT_IDS_TF, _SCENE_FEATS_TF)  # build
_mlp.set_weights([_w[f'w{i}'] for i in range(len(_w.files)-1)])

# 2. Compute scene features for new scene
new_scene_feats = compute_scene_features(
    freq_hz      = float(FREQUENCY_HZ),
    ndsm_data    = your_ndsm_array,   # loaded from ndsm.tif
    tx_pos       = tx_position,
    rx_positions = your_receivers,
)

# 3. Predict material parameters — NO RT needed
calib_mats = _mlp.predict_materials(new_scene_feats)
# Returns: {"concrete": {"er":x, "sigma":y, "scatter":z}, ...}

# 4. Apply to Sionna 2 scene
for mat_name, props in calib_mats.items():
    obj = scene.get(mat_name)
    obj.relative_permittivity  = props["er"]
    obj.conductivity           = props["sigma"]
    obj.scattering_coefficient = props["scatter"]
''')


---
## CELL 12 · TX Orientation Optimization

Optimises the transmitter antenna pointing direction (`tx.orientation`) to maximise coverage.

| Parameter | Value |
|-----------|-------|
| Optimizer | RMSprop |
| Loss | −E[log₂(1 + SNR × path_gain)] |
| Variable | `tx.orientation` (azimuth, tilt) as `tf.Variable` |
| Steps | `ORI_STEPS` |

Run after Cell 11b (material calibration) for best results.


In [ ]:
tx_name = list(scene.transmitters.keys())[0]
tx      = scene.transmitters[tx_name]

_ori_init = [0.0, 0.0, 0.0]
try: _ori_init = [_safe(tx.orientation[i]) for i in range(3)]
except: pass

tx.orientation = tf.Variable(_ori_init, dtype=tf.float32, name='tx_orientation')
print(f'TX "{tx_name}"  orientation = {_ori_init}  → tf.Variable')

def cm_capacity_loss(sc, cell_size=10.0, n_samp=ORI_NUM_SAMP):
    """Loss = −E[log₂(1 + SNR_scale × path_gain)]  (diff-rt Learning_Orientation)."""
    try:
        cm = sc.coverage_map(
            cm_cell_size        = cell_size,
            max_depth           = 3,
            num_samples         = n_samp,
            los                 = True,
            specular_reflection = True,
            diffuse_reflection  = False,
            refraction          = False,
            diffraction         = True,
        )
        pg = None
        for attr in ('path_gain', 'as_tensor'):
            if hasattr(cm, attr):
                val = getattr(cm, attr)
                pg  = val() if callable(val) else val
                break
        if pg is None: raise AttributeError('no path_gain')
        pg_flat  = tf.reshape(tf.cast(pg[0], tf.float32), [-1])
        capacity = tf.reduce_mean(
            tf.math.log(1.0 + SNR_SCALE * pg_flat) / tf.math.log(2.0))
        return -capacity, cm
    except Exception as e:
        print(f'  cm_capacity_loss error: {e}')
        return tf.constant(0.0), None

print(f'SNR_SCALE = {SNR_SCALE:.2e}')

In [ ]:
ori_optimizer = tf.keras.optimizers.RMSprop(learning_rate=ORI_LR)
ori_history   = {'step': [], 'rate_bit': [], 'orientation': []}

print(f'TX orientation optimization – {ORI_STEPS} steps  RMSprop LR={ORI_LR}')
print('-' * 60)

cm_before_np = None
t0 = time.time()
for step in range(ORI_STEPS):
    with tf.GradientTape() as tape:
        loss_val, cm_opt = cm_capacity_loss(scene, cell_size=10.0, n_samp=ORI_NUM_SAMP)

    if step == 0 and cm_opt is not None:
        cm_before_np = _cm_to_numpy(cm_opt)

    grads    = tape.gradient(loss_val, tape.watched_variables())
    valid_gv = [(g, v) for g, v in zip(grads, tape.watched_variables()) if g is not None]
    if valid_gv: ori_optimizer.apply_gradients(valid_gv)

    rate = float(-loss_val.numpy())
    ori  = list(tx.orientation.numpy())
    ori_history['step'].append(step)
    ori_history['rate_bit'].append(rate)
    ori_history['orientation'].append(ori)
    print(f'  step {step:3d}  rate={rate:.4f} bit  '
          f'ori=[{", ".join(f"{o:.3f}" for o in ori)}]  t={time.time()-t0:.0f}s', end='\r')

print()
print('-' * 60)
ori_final = list(tx.orientation.numpy())
print(f'Initial orientation  : {_ori_init}')
print(f'Optimized orientation: {[round(o, 4) for o in ori_final]}')
rate_initial = ori_history['rate_bit'][0]  if ori_history['rate_bit'] else 0
rate_final   = ori_history['rate_bit'][-1] if ori_history['rate_bit'] else 0
print(f'Rate improvement     : {rate_initial:.4f} → {rate_final:.4f} bit  (+{rate_final-rate_initial:.4f})')

---
## CELL 14 · CNN+MLP Path Loss Predictor — Sionna RT Hybrid Model

Combines physics-based Sionna RT simulation output with a data-driven CNN  
to correct residual errors that ray tracing cannot model (vegetation attenuation,  
diffraction edge effects, near-field clutter).

**Architecture:**

```
nDSM patch (64×64) ──► Conv2D(32,3) ──► Conv2D(64,3) ──► GlobalAvgPool ──► FC(64) ──┐
                                                                                       ├──► FC(64) ──► RSSI_pred
[rssi_sim, dist_km,  ──────────────────────────────────────────────────── FC(32) ──┘
 tx_h, rx_h, az_deg]
```

**Inputs:**

| Input | Shape | Source |
|-------|-------|--------|
| nDSM patch | (64, 64, 1) | `ndsm.tif` cropped 64m×64m centred on RX |
| `rssi_sim` | scalar | `rssi_sim_cached` from Cell 10b pre-trace |
| `dist_km` | scalar | TX–RX distance from receiver_locations.csv |
| `tx_h_m` | scalar | TX height AGL |
| `rx_h_m` | scalar | RX height AGL |
| `az_deg` | scalar | TX→RX azimuth bearing (0–360°) |

**Output:** Predicted RSSI (dBm) — trained against measured RSSI  
**Loss:** MSE on dBm  
**Split:** 80% train / 20% test (random seed fixed)


In [ ]:
# ====================================================================
# CELL 14 — CNN+MLP Residual Corrector (Sionna RT Hybrid Model)
#
# MIXING STRATEGY — Option B: Residual learning
#   RSSI_final = rssi_sim  +  CNN_residual
#
#   rssi_sim  : Sionna RT physics output (Cell 10b)
#   CNN_residual : what RT gets wrong — learned from nDSM + scalars
#
# The CNN never replaces RT — it only corrects its errors.
# Loss = MSE(residual_pred, residual_true)
#      = MSE(rssi_sim + residual_pred, rssi_meas)
# ====================================================================
import os, time
import numpy as np
import tensorflow as tf
import rasterio

# ── Config ────────────────────────────────────────────────────────────
NDSM_PATH   = os.path.join(BASE_DIR, 'ndsm.tif')
PATCH_M     = 64           # patch size in pixels (auto-scaled to metres below)
PATCH_HALF  = PATCH_M // 2
# nDSM pixel resolution read dynamically from rasterio — do NOT hardcode 1m/pixel
_ndsm_res_check = rasterio.open(NDSM_PATH)
_ndsm_pixel_m   = abs(_ndsm_res_check.transform.a)   # metres per pixel (x-axis)
_ndsm_res_check.close()
PATCH_M_metres  = int(PATCH_M * _ndsm_pixel_m)       # real-world size of patch
print(f'  nDSM resolution: {_ndsm_pixel_m:.2f} m/pixel  →  patch = {PATCH_M}px = {PATCH_M_metres}m')
CNN_EPOCHS  = 150
CNN_LR      = 1e-3
CNN_BATCH   = 32
TRAIN_SPLIT = 0.8
RANDOM_SEED = 42

print('=' * 70)
print('CELL 14 — CNN+MLP Residual Corrector (Sionna RT Hybrid)')
print('=' * 70)
print('  Strategy : RSSI_final = rssi_sim + CNN_residual')
print('  CNN task : predict (RSSI_meas - RSSI_sim) per receiver')
print()
# ── Derive scene origin in nDSM CRS from center_utm ─────────────────────────
_ndsm_tmp   = rasterio.open(NDSM_PATH)
_ndsm_epsg  = _ndsm_tmp.crs.to_epsg()
_ndsm_tmp.close()
_to_ndsm    = Transformer.from_crs(f'EPSG:{UTM_EPSG}', f'EPSG:{_ndsm_epsg}', always_xy=True)
scene_origin_x, scene_origin_y = _to_ndsm.transform(center_utm[0], center_utm[1])
print(f'  Scene origin EPSG:{_ndsm_epsg}: E={scene_origin_x:.1f}  N={scene_origin_y:.1f}')

# ── 1. Load nDSM ─────────────────────────────────────────────────────
print(f'Loading nDSM: {NDSM_PATH}')
_ndsm_src  = rasterio.open(NDSM_PATH)
_ndsm_arr  = _ndsm_src.read(1).astype(np.float32)
_ndsm_arr  = np.clip(_ndsm_arr, 0.0, 120.0) / 120.0   # normalised [0,1]
print(f'  shape={_ndsm_arr.shape}  CRS={_ndsm_src.crs}')

def _get_patch(local_x, local_y):
    """64×64 nDSM patch centred on receiver (local XY → BNG → raster row/col)."""
    bng_e = local_x + scene_origin_x
    bng_n = local_y + scene_origin_y
    row, col = _ndsm_src.index(bng_e, bng_n)
    H, W = _ndsm_arr.shape
    r0 = int(np.clip(row - PATCH_HALF, 0, H - PATCH_M))
    c0 = int(np.clip(col - PATCH_HALF, 0, W - PATCH_M))
    patch = _ndsm_arr[r0:r0+PATCH_M, c0:c0+PATCH_M]
    if patch.shape != (PATCH_M, PATCH_M):
        patch = np.pad(patch,
                       ((0, PATCH_M-patch.shape[0]),
                        (0, PATCH_M-patch.shape[1])),
                       constant_values=0.0)
    return patch.astype(np.float32)[..., np.newaxis]   # (64,64,1)

# ── 2. Build dataset ──────────────────────────────────────────────────
if 'rssi_sim_cached' not in dir():
    raise RuntimeError('rssi_sim_cached not found — run Cell 10b first')

_tx_pos = list(scene.transmitters.values())[0].position.numpy()
_tx_h   = float(_tx_pos[2])

print(f'\nBuilding dataset from {len(calib_receivers)} receivers ...')

patches   = []   # nDSM patches — CNN input
scalars   = []   # [dist_km, tx_h, rx_h, az_sin, az_cos] — MLP input
residuals = []   # RSSI_meas - RSSI_sim — regression target
sim_vals  = []   # rssi_sim per valid receiver — for final prediction

for i, (rx, rssi_sim_v, rssi_meas_v) in enumerate(
        zip(calib_receivers,
            rssi_sim_cached.numpy(),
            calib_rssi_meas.numpy())):

    if not np.isfinite(rssi_sim_v)  or rssi_sim_v  < -150.0: continue
    if not np.isfinite(rssi_meas_v) or rssi_meas_v < -150.0: continue

    rx_pos  = rx.position.numpy()
    dx, dy  = rx_pos[0] - _tx_pos[0], rx_pos[1] - _tx_pos[1]
    dist_km = np.sqrt(dx**2 + dy**2) / 1000.0
    rx_h    = float(rx_pos[2])
    az_rad  = np.arctan2(dx, dy)

    # Scalar features — normalised
    sc = np.array([
        dist_km   / 10.0,          # distance (max ~10 km in scene)
        _tx_h     / 100.0,         # TX height AGL
        rx_h      / 50.0,          # RX height
        float(np.sin(az_rad)),     # azimuth encoded as sin/cos
        float(np.cos(az_rad)),     #   avoids 0°/360° discontinuity
    ], dtype=np.float32)

    # Residual target = what RT got wrong
    residual = float(rssi_meas_v) - float(rssi_sim_v)

    patches.append(_get_patch(float(rx_pos[0]), float(rx_pos[1])))
    scalars.append(sc)
    residuals.append(residual)
    sim_vals.append(float(rssi_sim_v))

N = len(patches)
print(f'Valid pairs  : {N}')
print(f'Residual     : mean={np.mean(residuals):.2f} dB  std={np.std(residuals):.2f} dB')
print(f'              min={np.min(residuals):.1f}  max={np.max(residuals):.1f} dB')

X_patch = np.stack(patches)    # (N,64,64,1)
X_sc    = np.stack(scalars)    # (N,5)
y_res   = np.array(residuals, dtype=np.float32)   # (N,) — residual target
y_sim   = np.array(sim_vals,  dtype=np.float32)   # (N,) — RT prediction

# ── 3. Train / test split ─────────────────────────────────────────────
rng     = np.random.default_rng(RANDOM_SEED)
idx     = rng.permutation(N)
n_tr    = int(N * TRAIN_SPLIT)
tr, te  = idx[:n_tr], idx[n_tr:]

Xp_tr, Xp_te = X_patch[tr], X_patch[te]
Xs_tr, Xs_te = X_sc[tr],    X_sc[te]
yr_tr, yr_te = y_res[tr],   y_res[te]
ys_tr, ys_te = y_sim[tr],   y_sim[te]   # RT predictions on test set
print(f'Train : {len(tr)}   Test : {len(te)}')

# ── 4. Model — CNN branch + MLP branch → residual ────────────────────
def build_residual_model():
    # CNN branch: processes 64×64 nDSM height map
    img_in = tf.keras.Input(shape=(PATCH_M, PATCH_M, 1), name='ndsm_patch')
    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(img_in)
    x = tf.keras.layers.MaxPool2D(2)(x)                          # 32×32
    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPool2D(2)(x)                          # 16×16
    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)              # (128,)
    x = tf.keras.layers.Dense(64, activation='relu')(x)

    # MLP branch: processes scalar physics features
    sc_in = tf.keras.Input(shape=(5,), name='scalar_feats')
    s = tf.keras.layers.Dense(32, activation='relu')(sc_in)
    s = tf.keras.layers.Dense(32, activation='relu')(s)

    # Merge: CNN geometry + MLP physics → residual prediction
    merged = tf.keras.layers.Concatenate()([x, s])              # (96,)
    out    = tf.keras.layers.Dense(64, activation='relu')(merged)
    out    = tf.keras.layers.Dropout(0.2)(out)
    # Linear output — residual can be positive or negative dB
    out    = tf.keras.layers.Dense(1, activation='linear',
                                   name='residual_db')(out)
    return tf.keras.Model(inputs=[img_in, sc_in], outputs=out)

_CNN_WEIGHTS = os.path.join(OUTPUT_DIR, f'cnn_residual_{int(FREQUENCY_HZ/1e6)}mhz.h5')
model_cnn = build_residual_model()
if os.path.exists(_CNN_WEIGHTS):
    model_cnn.load_weights(_CNN_WEIGHTS)
    print(f'  Loaded existing CNN weights from {_CNN_WEIGHTS}')
    print('  Delete the file to force retraining.')
    _CNN_SKIP_TRAIN = True
else:
    _CNN_SKIP_TRAIN = False
model_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(CNN_LR),
    loss='mse',
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)
model_cnn.summary()

# ── 5. Training ────────────────────────────────────────────────────────
if not _CNN_SKIP_TRAIN:
    print(f'\nTraining residual corrector ({CNN_EPOCHS} epochs) ...')
t0 = time.time()

hist = model_cnn.fit(
    x=[Xp_tr, Xs_tr], y=yr_tr,
    validation_data=([Xp_te, Xs_te], yr_te),
    epochs=CNN_EPOCHS, batch_size=CNN_BATCH,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_rmse', patience=20, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5, verbose=0)
    ],
    verbose=0
)
best_ep = int(np.argmin(hist.history['val_rmse'])) + 1
print(f'Best epoch : {best_ep}  (of {len(hist.history["rmse"])} run)')
model_cnn.save(_CNN_WEIGHTS)
print(f'  CNN weights saved → {_CNN_WEIGHTS}')
print(f'Done in {time.time()-t0:.1f}s')

# Print every 10 epochs
for ep in range(0, len(hist.history['rmse']), 10):
    print(f'  epoch {ep+1:3d}  train={hist.history["rmse"][ep]:.2f} dB'
          f'  val={hist.history["val_rmse"][ep]:.2f} dB')

# ── 6. Final evaluation ───────────────────────────────────────────────
# Residual predictions on test set
resid_pred = model_cnn.predict([Xp_te, Xs_te], verbose=0).flatten()

# Final prediction = RT output + CNN residual correction
rssi_hybrid = ys_te + resid_pred
rssi_rt     = ys_te                    # RT-only baseline
rssi_meas_te = yr_te + ys_te           # recover measured from residual + sim

rmse_rt     = float(np.sqrt(np.mean((rssi_rt     - rssi_meas_te)**2)))
rmse_hybrid = float(np.sqrt(np.mean((rssi_hybrid - rssi_meas_te)**2)))
mae_hybrid  = float(np.mean(np.abs(rssi_hybrid - rssi_meas_te)))
bias_hybrid = float(np.mean(rssi_hybrid - rssi_meas_te))

print(f'\n{"Model":<30} {"RMSE":>8} {"MAE":>8} {"Bias":>8}  N={len(te)}')
print('-' * 65)
print(f'  RT only (Sionna Cell 10b) {rmse_rt:>8.2f} dB')
print(f'  RT + CNN residual         {rmse_hybrid:>8.2f} dB  {mae_hybrid:>8.2f}  {bias_hybrid:>+8.2f}')
print(f'  RMSE improvement          {rmse_rt - rmse_hybrid:>+8.2f} dB')
print()
print('Formula applied:')
print('  RSSI_final = RSSI_sim (Sionna RT) + CNN_residual (this model)')


---
## CELL 13 · Post-Calibration Analysis

Computes final coverage map and per-receiver error statistics using calibrated material properties.

**Outputs:**
- Final coverage map (calibrated vs pre-calibration)
- Per-receiver: RSSI_sim, RSSI_meas, error (dB), distance (m)
- Summary: RMSE, MAE, Bias, R² across all 1 200 receivers
- CSV export: `simulation_results_calibrated.csv`


In [ ]:
print('Post-calibration path computation ...')
print(f'  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

paths_cal = scene.compute_paths(
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_PS,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = True,
)
print('Done.')

a_np   = _to_numpy(paths_cal.a)
tau_np = _to_numpy(paths_cal.tau)
if a_np.ndim == 6: a_np = a_np[0]

n_rx    = a_np.shape[0]
power   = np.sum(np.abs(a_np)**2, axis=tuple(range(1, a_np.ndim)))
pg_cal  = power
pg_cal_db = 10 * np.log10(pg_cal + 1e-30)

print(f'Post-calibration path gain at {n_rx} receivers:')
print(f'  mean={pg_cal_db.mean():.1f} dB  min={pg_cal_db.min():.1f} dB  max={pg_cal_db.max():.1f} dB')

In [ ]:
records = []
for i, rx in enumerate(receivers[:n_rx]):
    x   = _safe(rx.position[0])
    y   = _safe(rx.position[1])
    z   = _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    pg_pre  = float(pg_at_rx_pre[i]) if i < len(pg_at_rx_pre) else float('nan')
    pg_post = float(pg_cal_db[i])    if i < len(pg_cal_db)    else float('nan')
    records.append({
        'receiver'    : rx.name,
        'lon'         : round(lon, 6),
        'lat'         : round(lat, 6),
        'x_m'         : round(x,   2),
        'y_m'         : round(y,   2),
        'z_m'         : round(z,   3),
        'pg_pre_db'   : round(pg_pre,  2),
        'pg_post_db'  : round(pg_post, 2),
        'delta_pg_db' : round(pg_post - pg_pre, 2) if not np.isnan(pg_pre) else float('nan'),
    })

df_out = pd.DataFrame(records)
out_csv = os.path.join(OUTPUT_DIR, 'receiver_results_calibrated.csv')
df_out.to_csv(out_csv, index=False)
print(f'Saved {len(df_out)} receivers to {out_csv}')
print(df_out.head(10).to_string(index=False))

In [ ]:
print('Computing final calibrated coverage map ...')
cm_final    = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = True,
)
cm_final_np    = _cm_to_numpy(cm_final)
pg_pre_db_2d   = 10 * np.log10(cm_pre_np[0]   + 1e-30)
pg_final_db_2d = 10 * np.log10(cm_final_np[0] + 1e-30)

vmin = min(np.nanpercentile(pg_pre_db_2d, 5),  np.nanpercentile(pg_final_db_2d, 5))
vmax = max(np.nanpercentile(pg_pre_db_2d, 99), np.nanpercentile(pg_final_db_2d, 99))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, data, title in [
    (axes[0], pg_pre_db_2d,   'Before Calibration (ITU defaults)'),
    (axes[1], pg_final_db_2d, 'After Calibration (diff-rt)'),
    (axes[2], pg_final_db_2d - pg_pre_db_2d, 'Δ Path Gain (After − Before)'),
]:
    if 'Δ' in title:
        im = ax.imshow(data, origin='lower', cmap='RdYlGn', vmin=-10, vmax=10)
    else:
        im = ax.imshow(data, origin='lower', cmap='jet', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Path Gain (dB)')
    ax.set_title(title); ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')

plt.suptitle('Coverage Map: Before vs After Differentiable RT Calibration', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nAll results saved to:', OUTPUT_DIR)